# MZM 소자 측정 데이터 분석 및 시각화 파이프라인

본 노트북은 **MZM(Mach-Zehnder Modulator)** 소자로부터 측정된 raw 데이터를 자동으로 파싱하고, 핵심 특성 파라미터(IL, ER, Vpi 등)를 분석 및 추출하여 **Wafer Map** 및 **Box Plot** 형태로 시각화하는 전체 공정을 포함하고 있습니다.
# 이 노트북을 실행하기 전 필요한 라이브러리 설치
# !pip install numpy pandas matplotlib scipy

## 📌 데이터 분석 파이프라인 개요
1. **데이터 전처리 & 파싱**: Raw 데이터 로드 및 정제 (`ref_poly`,`data_parser`,`plot`)
2. **소자 특성 피팅 & 분석**: 물리적 모델 기반 피팅 및 파라미터 추출 (`zoom`, `flatting`,`Fitting`, `Phase shift - V`, `VpiL`)
3. **핵심 지표 산출**: 삽입 손실(IL), 소멸비(ER), 구동 전압(Vpi) 분석 (`IL_Analysis`, `ER_Analysis`, `VpiL_Analysis`)
4. **시각화 및 요약**: 데이터 시각화 및 최종 보고서 내보내기 ( `combine_plot`,  `export_summary`)

### [1] (데이터 분석 및 노이즈 제거)
* **파일명**: `ref_poly, data_parser`
* **설명**:
* **ref_poly**:
* 1. 최대 최솟점 주변에서 3개의 점을 찾아 보정(2차피팅)
* 2. Ref 스펙트럼 3차피팅
* **data_parser**:
* 1. 수많은 하위 폴더에서 XML만 찾기(LMZO,LMZC,좌표,Data)
* 2. 바이어스에 따른 값과 Ref 분류 및 라벨정리


In [1]:
"""
- q_sub      : valley(또는 peak) 주변 3점 포물선 subpixel 보정
- ref_poly   : Reference 스펙트럼 3차 다항식 평탄화 기준선
"""

import numpy as np
from scipy.signal import savgol_filter

# [공통 분석 파라미터]
SAVGOL_WINDOW = 31     # 한번에 계산하는 데이터 개수
SAVGOL_POLY = 3        # 모든 데이터의 다항식 차수
REF_POLY_DEG = 3       # Reference 평탄화에 사용할 다항식 차수


def q_sub(x: np.ndarray, y: np.ndarray) -> float:
    # 1. 데이터 개수가 3개 미만이면 피팅이 불가능하므로 최소값 인덱스 바로 반환
    if len(x) < 3:
        return x[np.argmin(y)]

    idx = np.argmin(y)

    # 2. 최소값 위치가 경계면(처음 또는 끝)이면 주변 3점을 잡을 수 없으므로 이산(Discrete) 최소값 반환
    if idx == 0 or idx == len(x) - 1:
        return x[idx]

    # 3. 최소값 중심 주변 3점 추출 및 2차 다항식 피팅 (y = c[0]*x^2 + c[1]*x + c[2])
    c = np.polyfit(x[idx - 1 : idx + 2], y[idx - 1 : idx + 2], 2)

    # 4. 포물선 꼭지점 계산 (분모가 0에 가까운 일직선 케이스 예외 처리)
    return -c[1] / (2 * c[0]) if abs(c[0]) > 1e-12 else x[idx]


def ref_poly(ref_wl: np.ndarray, ref_il: np.ndarray, smooth: bool = False) -> np.poly1d:
    # 1. 노이즈 제거 여부에 따른 전처리 데이터 선택
    y = savgol_filter(ref_il, SAVGOL_WINDOW, SAVGOL_POLY) if smooth else ref_il

    # 2. 다항식 피팅 후 1차원 다항식 클래스(poly1d) 형태로 반환 (호출 시 바로 y값 계산 가능)
    return np.poly1d(np.polyfit(ref_wl, y, REF_POLY_DEG))

In [2]:
import os
import xml.etree.ElementTree as ET
import numpy as np
from datetime import datetime

def parse_wafer_data(data_path: str, target_wafers: list):
    """지정된 폴더를 탐색하여 XML 데이터를 파싱하고 다이(Die)별 측정 데이터를 반환합니다.
    Args:
        data_path (str): XML 파일들이 들어있는 최상위 폴더 경로
        target_wafers (list): 분석할 타겟 웨이퍼 이름 목록 (예: ['D07', 'D08'])

    Yields:
        dict: 웨이퍼 ID, 대역(Band), Die 좌표, 날짜, 참조(Ref) 및 바이어스(Bias) 측정 데이터가 포함된 딕셔너리
    """

    # 1. os.walk를 사용하여 폴더 내의 모든 파일을 하위 폴더까지 재귀적으로 탐색
    for root_dir, dirs, files in os.walk(data_path):
        for file_name in files:
            # XML 파일이 아니면 건너뜀
            if not file_name.lower().endswith('.xml'):
                continue

            file_path = os.path.join(root_dir, file_name)

            # 2. 파일 경로나 이름에 타겟 웨이퍼(D07, D08 등)가 포함되어 있는지 확인
            if not any(w in file_path for w in target_wafers):
                continue

            try:
                # 3. XML 파일 열기 및 파싱
                with open(file_path, 'r', encoding='utf-8') as f:
                    tree = ET.parse(f)
                    root = tree.getroot()

                    # TestSiteInfo 노드 확인(LMZO,C 확인 및 웨이퍼 좌표(n,m),Date)
                    info = root.find('.//TestSiteInfo')
                    if info is None:
                        continue

                    # 4. 소자 타입(TestSite)에 따른 파장(Wavelength) 대역 분류
                    test_site = info.get('TestSite', '')
                    if 'LMZO' in test_site.upper():
                        band, wl_min, wl_max, tgt_wl = 'LMZO', 1260.0, 1360.0, 1310.0
                    elif 'LMZC' in test_site.upper():
                        band, wl_min, wl_max, tgt_wl = 'LMZC', 1520.0, 1580.0, 1550.0
                    else:
                        continue  # 관심 없는 소자면 건너뜀

                    # 5. Die의 Row(행), Column(열) 좌표 추출
                    die_c = int(info.get('DieRow', 0)) if info.get('DieRow') else 0
                    die_r = int(info.get('DieColumn', 0)) if info.get('DieColumn') else 0

                    # 6. 웨이퍼 ID 추출 (target_wafers 목록 중 경로에 포함된 것 찾기)
                    wafer_id = next((w for w in target_wafers if w in file_path), "Unknown")

                    # 7. 측정 날짜 정보 추출 및 'YYYYMMDD' 포맷으로 통일화
                    date_raw = info.get('Date') or root.get('Date') or root.get('CreationDate')
                    if not date_raw:
                        date_elem = root.find('.//Date') or root.find('.//CreationDate')
                        if date_elem is not None:
                            date_raw = date_elem.text

                    date_str = "Unknown_Date"
                    if date_raw:
                        # 소수점 이하(밀리초 등) 제거 및 공백 제거
                        date_clean = date_raw.split('.')[0].strip()

                        # 장비마다 다를 수 있는 날짜 포맷 대응 리스트
                        date_formats = [
                            "%a %b %d %H:%M:%S %Y",
                            "%Y-%m-%d %H:%M:%S",
                            "%Y-%m-%d",
                            "%Y-%m-%dT%H:%M:%S",
                            "%Y/%m/%d %H:%M:%S"
                        ]

                        for fmt in date_formats:
                            try:
                                dt = datetime.strptime(date_clean, fmt)
                                date_str = dt.strftime("%Y%m%d")
                                break
                            except ValueError:
                                continue

                        # 정해진 포맷에 안 맞으면 특수문자만 제거하여 저장
                        if date_str == "Unknown_Date":
                            date_str = "".join(filter(str.isalnum, date_clean))

                    # 8. 파장 스윕(WavelengthSweep) 데이터 추출
                    sweeps = root.findall('.//WavelengthSweep')
                    if not sweeps or len(sweeps) < 2:
                        continue

                    ref_data = None
                    bias_list = []

                    # 9. 각 Sweep 데이터(파장, 손실값, 인가 전압 등)를 Numpy 배열로 변환
                    for i, sweep in enumerate(sweeps):
                        l_elem, il_elem = sweep.find('L'), sweep.find('IL')
                        if l_elem is None or il_elem is None:
                            continue

                        # 문자열 형태의 데이터를 float형 Numpy 배열로 변환 (속도 및 연산 이점)
                        l_data = np.array(list(map(float, l_elem.text.split(','))))
                        il_data = np.array(list(map(float, il_elem.text.split(','))))
                        bias_str = sweep.get('DCBias', '')

                        # 마지막 Sweep을 Reference(기준) 데이터로 취급
                        if i == len(sweeps) - 1:
                            ref_data = {'wl': l_data, 'il': il_data, 'label': 'REF'}
                        else:
                            # 전압(Bias) 값이 있는 경우 리스트에 추가
                            try:
                                bias_val = float(bias_str)
                                label = f'{bias_val}V'
                            except:
                                bias_val = None
                                label = f'Bias: {bias_str}'
                            bias_list.append({'bias': bias_val, 'wl': l_data, 'il': il_data, 'label': label})

                    # Reference 데이터가 없으면 유효하지 않으므로 건너뜀
                    if ref_data is None:
                        continue

                    # 10. 최종 정리된 1개의 Die 데이터를 딕셔너리 형태로 반환 (Generator)
                    yield {
                        'wafer_id': wafer_id,
                        'band': band,
                        'die_c': die_c,
                        'die_r': die_r,
                        'wl_min': wl_min,
                        'wl_max': wl_max,
                        'target_wl': tgt_wl,
                        'date': date_str,
                        'ref_data': ref_data,
                        'bias_data_list': bias_list
                    }

            except Exception as e:
                print(f"[{file_path}] 파싱 에러: {e}")

### 🖼️ [2] 단일 데이터 시각화 (Plot)
* **파일명**: `plot.py`
* **설명**: 개별 소자 혹은 특정 다이(Die)의 파장별 투과 특성 곡선 및 전압 특성 곡선을 시각화고 저장합니다.

In [3]:
import matplotlib
import matplotlib.pyplot as plt

# Jupyter에서도 GUI 충돌 방지
matplotlib.use("Agg")

def _draw_and_save_plot(d, base_save_dir):
    fig, ax = plt.subplots(figsize=(10, 6))

    try:
        # Bias 데이터 플롯
        for b in d['bias_data_list']:
            wl = np.asarray(b['wl'])
            il = np.asarray(b['il'])

            if len(wl) == 0 or len(il) == 0:
                continue

            if np.isnan(il).all():
                print(f"NaN only: {d['wafer_id']} C{d['die_c']} R{d['die_r']} {d['band']}")
                continue

            ax.plot(wl, il, label=b['label'], linewidth=2.5)

        # Reference 데이터 플롯
        ref_wl = np.asarray(d['ref_data']['wl'])
        ref_il = np.asarray(d['ref_data']['il'])

        if len(ref_wl) > 0 and not np.isnan(ref_il).all():
            ax.plot(
                ref_wl, ref_il,
                label=d['ref_data']['label'],
                linewidth=2.5, color='black',alpha=0.8, linestyle='--')

        # 그래프 서식 설정
        ax.set_title(
            f"Wafer: {d['wafer_id']} / Coord: ({d['die_c']}, {d['die_r']}) / Band: {d['band']}",
            fontsize=18, fontweight='bold', pad=15
        )
        ax.set_xlabel("Wavelength (nm)", fontsize=16, fontweight='bold')
        ax.set_ylabel("Transmission (dB)", fontsize=16, fontweight='bold')
        ax.set_ylim(-65, 5)
        ax.tick_params(axis='both', labelsize=13, width=2)
        ax.legend(loc='best', prop={'size': 12, 'weight': 'bold'})
        ax.grid(True, linestyle='--', alpha=0.6, linewidth=1)

        # 축 두께 설정
        for spine in ax.spines.values():
            spine.set_linewidth(2)

        fig.tight_layout()

        # 저장 경로 및 파일명 설정
        date_str = d.get('date', 'Unknown_Date')
        w_dir = os.path.join(base_save_dir, d['wafer_id'], date_str)
        os.makedirs(w_dir, exist_ok=True)

        save_filename = f"{d['wafer_id']}_C{d['die_c']}_R{d['die_r']}_{d['band']}_Raw.png"

        # 이미지 저장
        fig.savefig(os.path.join(w_dir, save_filename), bbox_inches='tight', dpi=150)

        return save_filename

    finally:
        # 메모리 누수 방지를 위해 figure 닫기
        plt.close(fig)

# 실행 영역
zip_path = "./dat/HY202103"
base_save_dir = "./res/png"
target_wafers = ['D07', 'D08', 'D23', 'D24']

print("▶ 데이터 파싱 시작")

# 주의: parse_wafer_data 함수는 다른 셀이나 파일에서 미리 import 되어 있어야 합니다.
parsed_data_list = list(parse_wafer_data(zip_path, target_wafers))

total_tasks = len(parsed_data_list)
print(f"▶ 총 {total_tasks}개 그래프")

for i, d in enumerate(parsed_data_list, 1):
    _draw_and_save_plot(d, base_save_dir)

    # 진행 상황 출력
    if i % 50 == 0:
        print(f"진행: {i}/{total_tasks}")

print(f"✅ 총 {total_tasks}개 플롯 저장 완료")

▶ 데이터 파싱 시작
▶ 총 98개 그래프
진행: 50/98
✅ 총 98개 플롯 저장 완료


### 📐 [3] 데이터 평탄화 (Flatting)
* **파일명**: `flatting.py`
* **설명**: 측정 장비의 Ref에 속한 Grating Coupler를 제거하는 1차 평탄화와 소자내부의 노이즈를 제거하는 2차 피팅을 진행하여 데이터를 평탄화하는 전처리 작업을 수행하고 저장합니다.

In [4]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
# 주피터 환경 등에서 백그라운드로 그래프를 그리기 위한 설정
matplotlib.use("Agg")

def _draw_and_save_flat(d, base_save_dir):
    """
    광학 스펙트럼 데이터의 기준선(Reference) 곡률을 제거하여 평탄화(Flattening)한 뒤
    그래프를 그리고 파일로 저장하는 함수입니다.
    """

    # 1. Reference 데이터 추출 및 마스킹
    m = (d['ref_data']['wl'] >= d['wl_min']) & (d['ref_data']['wl'] <= d['wl_max'])
    v_ref_wl = d['ref_data']['wl'][m]
    v_ref_il = d['ref_data']['il'][m]

    # 다항식 피팅(polyfit) 최소 조건(데이터 4개 이상) 및 결측치 방어
    if len(v_ref_wl) < 4 or np.isnan(v_ref_il).all():
        return None

    # 위쪽 셀에 정의해둔 ref_poly 함수를 바로 사용합니다.
    poly = ref_poly(v_ref_wl, v_ref_il, smooth=False)

    # 2. 객체 지향 방식(fig, ax) 캔버스 생성
    fig, ax = plt.subplots(figsize=(10, 6))
    curve_count = 0  # 유효하게 그려진 선의 개수 카운트

    try:
        # 3. Bias 데이터 평탄화 및 플로팅
        for b in d['bias_data_list']:
            m_b = (b['wl'] >= d['wl_min']) & (b['wl'] <= d['wl_max'])
            v_wl = b['wl'][m_b]
            v_il = b['il'][m_b]

            # 결측치 방어
            if len(v_wl) == 0 or len(v_il) == 0 or np.isnan(v_il).all():
                continue

            # 3-1. [1차 평탄화] 실제 측정값에서 Reference 곡률(poly) 제거
            flat_il = v_il - poly(v_wl)

            # 3-2. [2차 평탄화] 선형 보정 (기울기 제거)
            peaks, _ = find_peaks(flat_il, prominence=3.0, distance=30)
            if len(peaks) >= 2:
                linear_fit = np.poly1d(np.polyfit(v_wl[peaks], flat_il[peaks], 1))
                flat_il = flat_il - linear_fit(v_wl)

            # 3-3. 평탄화된 데이터 선 그리기
            ax.plot(v_wl, flat_il, label=b['label'], alpha=0.8, linewidth=2.5)
            curve_count += 1

        # 4. Reference 데이터 평탄화 비교 플로팅
        ref_flat = v_ref_il - poly(v_ref_wl)
        ax.plot(v_ref_wl, ref_flat, label='REF',
                linewidth=2.5, color='black', alpha=0.8, linestyle='--')
        curve_count += 1

        if curve_count == 0:
            return None

        # 5. 그래프 서식 (스타일링) 설정
        ax.set_title(
            f"Wafer: {d['wafer_id']} / Coord: ({d['die_c']}, {d['die_r']}) / Band: {d['band']} Flattened",
            fontsize=18, fontweight='bold', pad=15
        )
        ax.set_xlabel('Wavelength (nm)', fontsize=16, fontweight='bold')
        ax.set_ylabel('Transmission (dB)', fontsize=16, fontweight='bold')

        # y=0 위치 가이드라인
        ax.axhline(0, color='gray', linestyle='--', linewidth=2)

        ax.set_xlim(d['wl_min'], d['wl_max'])
        ax.set_ylim(-65, 5)

        ax.tick_params(axis='both', labelsize=13, width=2)
        ax.grid(True, linestyle='--', alpha=0.6, linewidth=1)

        for spine in ax.spines.values():
            spine.set_linewidth(2)

        handles, labels = ax.get_legend_handles_labels()
        if len(handles) > 0:
            ax.legend(loc='best', prop={'size': 12, 'weight': 'bold'})

        fig.tight_layout()

        # 6. 폴더 생성 및 이미지 저장
        date_str = d.get('date', 'Unknown_Date')
        save_dir = os.path.join(base_save_dir, d['wafer_id'], date_str)
        os.makedirs(save_dir, exist_ok=True)

        filename = f"{d['wafer_id']}_C{d['die_c']}_R{d['die_r']}_{d['band']}_Flat.png"
        fig.savefig(os.path.join(save_dir, filename), bbox_inches='tight', dpi=150)

        return filename

    except Exception as e:
        print(f"오류 발생: {d['wafer_id']} C{d['die_c']} R{d['die_r']} {d['band']} -> {e}")
        return None

    finally:
        # 무조건 캔버스를 닫아 메모리 누수 방지
        plt.close(fig)

# 실행부 (Jupyter 등에서 순차 실행)

zip_path = "./dat/HY202103"
base_save_dir = "./res/png"
target_wafers = ['D07', 'D08', 'D23', 'D24']

print("▶ 데이터 파싱을 시작합니다...")

# 위쪽 셀에 있는 parse_wafer_data 함수를 그대로 사용
parsed_data_list = list(parse_wafer_data(zip_path, target_wafers))
total_items = len(parsed_data_list)

print(f"▶ 기본 평탄화 플롯 생성 시작 ({total_items}개)")

success_count = 0

for i, d in enumerate(parsed_data_list, 1):
    result = _draw_and_save_flat(d, base_save_dir)

    if result is not None:
        success_count += 1

    if i % 50 == 0:
        print(f"진행: {i}/{total_items}")

print(f"✅ 평탄화 이미지 저장 완료 (성공: {success_count}개 / 총 {total_items}개)")

▶ 데이터 파싱을 시작합니다...
▶ 기본 평탄화 플롯 생성 시작 (98개)
진행: 50/98
✅ 평탄화 이미지 저장 완료 (성공: 98개 / 총 98개)


### 🔍 [4] 특정 영역 확대 분석 (Zoom)
* **파일명**: `zoom.py`
* **설명**: -2.0(특정 바이어스)를 기준으로 LMZC(1550nm),LMZO(1310nm)주위의 가장가까운 최솟값을 찾고 확대하고, 저장합니다.

In [5]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# 주피터 환경에서 백그라운드 그리기를 활성화하여 메모리 충돌 및 브라우저 다운 방지
matplotlib.use("Agg")

# 주의: parse_wafer_data 와 ref_poly 함수는 위쪽 셀에서
# 이미 실행되어 메모리에 로드되어 있어야 합니다.
def _draw_and_save_zoom_only(d, base_save_dir):
    """
    특정 전압(-2V) 데이터의 피크를 분석하여 소자의 핵심 동작 파장 영역을 자동 포착(Zoom)하고,
    해당 구간만 확대된 평탄화 그래프를 그리며 파일로 저장하는 함수입니다.
    """
    # 1. Reference 데이터 추출 및 3차 다항식 피팅 (기본 평탄화 준비)
    # 설정된 전체 파장 범위(wl_min ~ wl_max) 데이터만 필터링
    m = (d['ref_data']['wl'] >= d['wl_min']) & (d['ref_data']['wl'] <= d['wl_max'])
    v_ref_wl = d['ref_data']['wl'][m]
    v_ref_il = d['ref_data']['il'][m]

    # 다항식 피팅을 위한 최소 데이터 개수 체크 및 결측치 방어
    if len(v_ref_wl) < 4 or np.isnan(v_ref_il).all():
        return None
    # [공용 함수 활용] 이미 로드된 ref_poly 함수를 사용하여 Reference의 3차 다항식 곡선 모델 생성
    poly = ref_poly(v_ref_wl, v_ref_il, smooth=False)

    # 2. 자동 줌(Zoom) 파장 구간 계산 알고리즘
    # 기본값은 전체 파장 범위로 설정 (피크를 못 찾을 경우를 대비한 안전장치)
    z_min = d['wl_min']
    z_max = d['wl_max']

    # 2-1. 줌 기준 전압 선택: 우선적으로 -2.0V 데이터를 찾고, 없으면 첫 번째 전압 데이터를 사용
    t_bias = next((b for b in d['bias_data_list'] if b['bias'] == -2.0),
                  d['bias_data_list'][0] if d['bias_data_list'] else None)

    if t_bias is not None:
        # 기준 전압 데이터 파장 마스킹
        mt = (t_bias['wl'] >= d['wl_min']) & (t_bias['wl'] <= d['wl_max'])
        wt = t_bias['wl'][mt]
        it = t_bias['il'][mt]

        if len(wt) > 0:
            # 1차 평탄화(it - poly(wt))를 적용한 상태에서 상단 봉우리(Peak)들을 검출
            peaks, _ = find_peaks(it - poly(wt), prominence=3.0, distance=30)

            # 봉우리가 최소 2개 이상 검출되어야 그 사이 구간으로 줌이 가능함
            if len(peaks) >= 2:
                # 이웃한 두 피크 사이의 정중앙 지점(Center) 배열 계산
                cent = (wt[peaks[:-1]] + wt[peaks[1:]]) / 2.0

                # 2-2. 소자 밴드별(O-band 또는 C-band) 타겟 중심 파장 설정
                band_str = str(d.get('band', '')).upper()
                if 'O' in band_str:
                    target_wl = d.get('target_wl', 1310.0)  # O밴드 기준 파장 (1310nm)
                else:
                    target_wl = d.get('target_wl', 1550.0)  # C밴드 기준 파장 (1550nm)

                # 2-3. 계산된 피크 중앙점들 중 타겟 중심 파장과 가장 가까운 인덱스 찾기
                idx = np.argmin(np.abs(cent - target_wl))

                # 인덱스 초과 방지 안전장치 속에서 최종 줌 구간(z_min, z_max) 확정
                # 좌우 피크 위치에서 각각 0.5nm 만큼 여유 마진을 주어 보기 좋게 만듦
                if idx + 1 < len(peaks):
                    z_min = wt[peaks[idx]] - 0.5
                    z_max = wt[peaks[idx + 1]] + 0.5

    # 3. 객체 지향 방식(fig, ax) 캔버스 생성 및 도화지 준비
    fig, ax = plt.subplots(figsize=(10, 6))
    curve_count = 0  # 실제 그려진 라인 개수 체크용

    try:
        # 4. 각 전압(Bias) 데이터 평탄화 및 그래프 작도
        for b in d['bias_data_list']:
            mb = (b['wl'] >= d['wl_min']) & (b['wl'] <= d['wl_max'])
            wb = b['wl'][mb]
            ib = b['il'][mb]

            # 데이터가 유효하지 않으면 패스
            if len(wb) == 0 or np.isnan(ib).all():
                continue
            # 4-1. 1차 평탄화: Reference 다항식 빼기
            flat = ib - poly(wb)
            # 4-2. 2차 평탄화: 잔여 기울기 보정
            # 평탄화 스펙트럼에서 피크를 찾고, 피크 간 선형 피팅(1차 직선)을 이용해 미세 기울기를 완전히 제거
            peaks_b, _ = find_peaks(flat, prominence=3.0, distance=30)
            if len(peaks_b) >= 2:
                linear_fit = np.poly1d(np.polyfit(wb[peaks_b], flat[peaks_b], 1))
                flat -= linear_fit(wb)

            # 4-3. 선 두께(2.5)와 반투명도(0.8)를 적용해 플롯 추가
            ax.plot(wb, flat, label=b['label'], alpha=0.8, linewidth=2.5)
            curve_count += 1

        # 만약 유효한 데이터가 없어서 그려진 곡선이 없다면 저장하지 않고 종료
        if curve_count == 0:
            return None

        # 5. 그래프 표준 서식 및 스타일링 적용 (Standard Style)
        # y=0 위치에 평탄화 기준선 역할을 할 회색 점선 표시
        ax.axhline(0, color='gray', linestyle='--', linewidth=2)

        # [핵심] 아까 위에서 치열하게 계산해낸 줌 구간(z_min ~ z_max)을 가로축 범위로 강제 지정!
        ax.set_xlim(z_min, z_max)
        ax.set_ylim(-65, 5)

        # 타이틀 및 축 이름 설정 (굵고 크게)
        ax.set_title(f"Wafer: {d['wafer_id']} / Coord: ({d['die_c']}, {d['die_r']}) / Band: {d['band']} Zoom Only",
                     fontsize=18, fontweight='bold', pad=15)
        ax.set_xlabel('Wavelength (nm)', fontsize=16, fontweight='bold')
        ax.set_ylabel('Transmission (dB)', fontsize=16, fontweight='bold')

        # 축 눈금 글꼴 및 선 두께 지정
        ax.tick_params(axis='both', labelsize=13, width=2)

        # 격자 점선 및 테두리(Spines) 두께 설정
        ax.grid(True, linestyle='--', alpha=0.6, linewidth=1)
        for spine in ax.spines.values():
            spine.set_linewidth(2)

        # 범례(Legend) 최적 위치 자동 배치 및 글꼴 설정
        handles, labels = ax.get_legend_handles_labels()
        if len(handles) > 0:
            ax.legend(loc='best', prop={'size': 12, 'weight': 'bold'})

        # 여백 제거 자동 조절
        fig.tight_layout()

        # 6. 저장 폴더 생성 및 고해상도 이미지 저장
        date_str = d.get('date', 'Unknown_Date')
        save_dir = os.path.join(base_save_dir, d['wafer_id'], date_str)
        os.makedirs(save_dir, exist_ok=True)

        # 파일명 생성 규칙 예시: D07_C1_R2_LMZO_Zoom.png
        filename = f"{d['wafer_id']}_C{d['die_c']}_R{d['die_r']}_{d['band']}_Zoom.png"

        # 선명한 화질(dpi=150)로 파일 저장
        fig.savefig(os.path.join(save_dir, filename), bbox_inches='tight', dpi=150)

        return filename

    except Exception as e:
        # 에러 발생 시 프로그램이 멈추지 않도록 로그만 출력하고 계속 진행
        print(f"오류 발생: {d['wafer_id']} C{d['die_c']} R{d['die_r']} {d['band']} -> {e}")
        return None

    finally:
        # [철통 보안] 정상 작동하든 에러가 나든 메모리가 누수되지 않도록 도화지를 확실히 파괴
        plt.close(fig)

# 7. 주피터 노트북 순차 실행 영역
zip_path = "./dat/HY202103"
base_save_dir = "./res/png"
target_wafers = ['D07', 'D08', 'D23', 'D24']

print("▶ 데이터 파싱을 시작합니다...")

# 위쪽 셀에 있는 parse_wafer_data 함수를 실행하여 리스트 변환
parsed_data_list = list(parse_wafer_data(zip_path, target_wafers))
total_items = len(parsed_data_list)

print(f"▶ 줌 전용 플롯 생성 시작 ({total_items}개)")

success_count = 0

# 순차적으로 데이터 카드를 하나씩 꺼내 그래프 생성
for i, d in enumerate(parsed_data_list, 1):
    result = _draw_and_save_zoom_only(d, base_save_dir)

    if result is not None:
        success_count += 1

    # 50개마다 콘솔창에 진행 상황 브리핑
    if i % 50 == 0:
        print(f"진행: {i}/{total_items}")

print(f"✅ 줌 전용 이미지 저장 완료 (성공: {success_count}개 / 총 {total_items}개)")

▶ 데이터 파싱을 시작합니다...
▶ 줌 전용 플롯 생성 시작 (98개)
진행: 50/98
✅ 줌 전용 이미지 저장 완료 (성공: 98개 / 총 98개)


### 🔮 [5] 데이터 피팅 (Fitting)
* **파일명**: `Fitting.py`
* **설명**: ZOOM을 진행한 후 관련된 리플을 제거하기 위하여 여러가지 다항식 피팅을 진행 후, 원본 데이터와 예측한 데이터 값으로 R^2값을 구하여 저장한다.

In [6]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter

# 주피터 환경에서 백그라운드 그리기를 활성화하여 메모리 누수 및 브라우저 다운을 방지하는 필수 설정
matplotlib.use("Agg")

def _draw_and_save_zoomed_plot(d, base_save_dir):
    """
    이전 셀에서 정의된 공용 ref_poly 함수를 그대로 가져와 호환성을 유지하며 Reference 곡률을 추출하고,
    회귀 정밀도(R²) 산출 및 핵심 파장 영역 확대를 거쳐 평탄화 스펙트럼 이미지를 저장합니다.
    """
    try:
        # 원본 데이터 카드에서 사용자가 지정한 유효 파장 대역(wl_min ~ wl_max)만 선별하여 슬라이싱합니다.
        m = (d['ref_data']['wl'] >= d['wl_min']) & (d['ref_data']['wl'] <= d['wl_max'])
        v_ref_wl = d['ref_data']['wl'][m]
        v_ref_il = d['ref_data']['il'][m]

        # 데이터 개수가 노이즈 필터 윈도우 크기(31)보다 적으면 연산이 불가능하므로 사전에 차단합니다.
        if len(v_ref_wl) < 31:
            return None

        # 입력 데이터가 전부 비어있는 NaN 값이라면 분석 대상에서 제외하고 패스합니다.
        if np.isnan(v_ref_il).all():
            return None

        # 평가 지표(R²) 계산과 최종 잔차 그래프 작도를 위해 노이즈가 제거된 스무딩 데이터를 확보합니다.
        sm_ref = savgol_filter(v_ref_il, 31, 3)

        # 이미 위에서 리플을 제거한 sm_ref를 강도 데이터로 직접 전달하고, smooth=False로 기존 함수를 호출합니다.
        poly = ref_poly(v_ref_wl, sm_ref, smooth=False)

        # 수학적으로 피팅된 다항식 모델의 Y 기반 기준선 예측값을 계산합니다.
        y_fitted = poly(v_ref_wl)

        # 모델의 설명력(R-squared) 산출의 기초가 되는 편차 잔차 제곱합을 구합니다.
        ss_res = np.sum((sm_ref - y_fitted) ** 2)

        # 결정계수 수식의 분모가 되는 전체 데이터의 편차 제곱합을 구합니다.
        ss_tot = np.sum((sm_ref - np.mean(sm_ref)) ** 2)

        # 완전한 직선 데이터가 들어와 ss_tot가 0이 되는 돌발 오류 상황을 안전하게 방어합니다.
        if ss_tot == 0:
            r_squared = 1.0
        else:
            r_squared = 1 - (ss_res / ss_tot)

        # 차트 요소를 정밀하게 그려낼 단독 캔버스 인스턴스(가로 10, 세로 6)를 생성합니다.
        fig, ax = plt.subplots(figsize=(10, 6))

        # 확대(Zoom) 범위를 기록할 가로축 변수를 선언하며, 기본값은 전체 대역으로 지정합니다.
        z_min = d['wl_min']
        z_max = d['wl_max']

        # 자동 줌 범위의 기준 삼을 전압 데이터(-2.0V)를 찾고, 부재 시 리스트의 첫 데이터를 선택합니다.
        tgt_bias = next(
            (b for b in d['bias_data_list'] if b['bias'] == -2.0),
            d['bias_data_list'][0] if d['bias_data_list'] else None
        )

        if tgt_bias is not None:
            # 타겟 바이어스 전압의 스펙트럼에서 유효 분석 파장 범위만 필터링합니다.
            m_t = (tgt_bias['wl'] >= d['wl_min']) & (tgt_bias['wl'] <= d['wl_max'])
            w_t = tgt_bias['wl'][m_t]
            i_t = tgt_bias['il'][m_t]

            if len(w_t) >= 31:
                # 줌 기준 전압 스펙트럼의 리플을 제거한 뒤, 불러온 poly 함수로 1차 평탄화를 시도합니다.
                flat_t = savgol_filter(i_t, 31, 3) - poly(w_t)

                # 1차 보정 데이터 상에서 봉우리(Peak)들을 검출하여 위치 인덱스를 반환받습니다.
                peaks, _ = find_peaks(flat_t, prominence=3.0, distance=30)

                # 최소 2개 이상의 봉우리가 식별되어야 채널 간 중점 범위를 기반으로 줌 영역 설정이 가능합니다.
                if len(peaks) >= 2:
                    # 이웃한 피크들 사이의 정중앙 지점들을 계산하여 배열로 매칭합니다.
                    centers = (w_t[peaks[:-1]] + w_t[peaks[1:]]) / 2.0

                    # 분석 중인 소자의 밴드 속성을 판별하기 위해 문자열을 대문자로 규격화합니다.
                    band_str = str(d.get('band', '')).upper()

                    # O밴드 소자는 1310nm, 그 외 C/L밴드 소자는 1550nm를 목표 관심 파장으로 정의합니다.
                    if 'O' in band_str:
                        target_wl = d.get('target_wl', 1310.0)
                    else:
                        target_wl = d.get('target_wl', 1550.0)

                    # 연산된 중점들 가운데 목표 관심 파장과 물리적으로 가장 가까운 인덱스를 추출합니다.
                    idx = np.argmin(np.abs(centers - target_wl))

                    # 인덱스 초과 에러 방지 안전 가드 하에 최종 돋보기 확대 가로축 범위(xlim)를 확정합니다.
                    # 채널의 양측 피크 위치 경계에서 외곽으로 각각 0.5nm의 시각적 마진을 더해 시원하게 연출합니다.
                    if idx + 1 < len(peaks):
                        z_min = w_t[peaks[idx]] - 0.5
                        z_max = w_t[peaks[idx + 1]] + 0.5

        # 도화지에 실제 그려질 유효 플롯의 라인 개수를 체크할 카운터를 선언합니다.
        curve_count = 0

        # 소자 카드 내부에 저장된 전압별 바이어스 데이터를 하나씩 순회하며 시각화합니다.
        for b in d['bias_data_list']:
            m_b = (b['wl'] >= d['wl_min']) & (b['wl'] <= d['wl_max'])
            v_wl = b['wl'][m_b]
            v_il = b['il'][m_b]

            # 데이터 유효 규격 조건에 미달하는 불량 전압 데이터 성분은 패스합니다.
            if len(v_wl) < 31:
                continue

            # 해당 전압 데이터가 결측치로 전락해 있다면 계산에서 제외합니다.
            if np.isnan(v_il).all():
                continue

            # 1차 평탄화: 전압 데이터의 리플을 정돈하고, 주피터 커널에 로드된 poly 모델로 기저 신호를 차감합니다.
            flat_il = savgol_filter(v_il, 31, 3) - poly(v_wl)

            # 2차 평탄화: 파장 전반에 잔존하여 스펙트럼을 뒤트는 선형 미세 기울기 성분을 강제 수평 보정합니다.
            peaks_b, _ = find_peaks(flat_il, prominence=3.0, distance=30)
            if len(peaks_b) >= 2:
                # 감지된 정점들을 잇는 1차 직선 추세선을 산출합니다.
                linear_fit = np.poly1d(np.polyfit(v_wl[peaks_b], flat_il[peaks_b], 1))
                # 1차 완성본 데이터에서 추세선 기울기 성분을 빼주어 완벽한 평행 상태를 구현합니다.
                flat_il -= linear_fit(v_wl)

            # 표준 규격 선 두께(2.5)와 투명도 옵션을 주어 전압별 최종 평탄화 곡선을 드로잉합니다.
            ax.plot(v_wl, flat_il, label=b['label'], linewidth=2.5, alpha=0.8)
            curve_count += 1

        # 다항식 피팅 회귀 정밀도가 표시된 Reference 자체의 최종 잔차 스펙트럼을 검은색 점선으로 레이어 병합합니다.
        ax.plot(
            v_ref_wl, sm_ref - y_fitted,
            label=f'REF (R²={r_squared:.4f})',
            color='black', linewidth=2.5, linestyle='--', zorder=10
        )
        curve_count += 1

        # 유효한 곡선이 단 하나도 생성되지 않은 불량 다이 데이터라면 파일 생성을 전면 무효화합니다.
        if curve_count == 0:
            plt.close(fig)
            return None

        # 소자 정보, 측정 다이 위치, 처리 유형이 명시된 종합 타이틀 텍스트를 출력합니다.
        ax.set_title(
            f"Wafer: {d['wafer_id']} / Coord: ({d['die_c']}, {d['die_r']}) / Band: {d['band']}\n"
            f"Smoothed, Flattened & Zoomed (via global ref_poly)",
            fontsize=18, fontweight='bold', pad=15
        )
        ax.set_xlabel('Wavelength (nm)', fontsize=16, fontweight='bold')
        ax.set_ylabel('Transmission (dB)', fontsize=16, fontweight='bold')

        # 평탄화 스펙트럼 판독의 평행 기준선 역할을 담당할 y=0 가이드 라인을 생성합니다.
        ax.axhline(0, color='gray', linestyle='--', alpha=0.6, linewidth=2)

        # 위쪽 알고리즘 섹션에서 세밀하게 추출해 둔 최적의 타겟 채널 확대 범위로 가로 범위를 한정합니다.
        ax.set_xlim(z_min, z_max)

        # 판독 능력을 고양할 보조 격자망 레이아웃을 투명 점선 스타일로 배치합니다.
        ax.grid(True, linestyle='--', alpha=0.6, linewidth=1)

        # 그래프 외곽 사각 테두리 선의 두께를 볼드하게 조정하여 완성도를 보강합니다.
        for spine in ax.spines.values():
            spine.set_linewidth(2)

        # 차트 내부 요소들의 라벨을 취합하고 비어있는 가장 이상적인 자리에 글꼴 스타일을 부여해 범례를 표시합니다.
        handles, labels = ax.get_legend_handles_labels()
        if len(handles) > 0:
            ax.legend(loc='best', prop={'size': 12, 'weight': 'bold'})

        # 여백 낭비 및 글자 잘림 불상사를 방지하는 레이아웃 리사이징 자동 기능입니다.
        fig.tight_layout()

        # 고유 측정 날짜 문자열 정보 데이터를 빌드하여 물리 디렉토리를 자동 생성합니다.
        date_str = d.get('date', 'Unknown_Date')
        save_dir = os.path.join(base_save_dir, d['wafer_id'], date_str)
        os.makedirs(save_dir, exist_ok=True)

        # 파일 명명 규칙에 의거하여 확장자를 매칭합니다. (예: D07_C1_R2_LMZO_Fitting.png)
        filename = f"{d['wafer_id']}_C{d['die_c']}_R{d['die_r']}_{d['band']}_Fitting.png"

        # 지정된 품질(150 dpi) 속성 가이드를 충족하여 하드디스크에 물리 파일로 저장합니다.
        fig.savefig(os.path.join(save_dir, filename), bbox_inches='tight', dpi=150)

        # 활성화된 캔버스 객체를 강제 파괴하여 주피터 백그라운드 구동에 따른 메모리 점유 현상을 원천 봉쇄합니다.
        plt.close(fig)

        return filename

    except Exception as e:
        # 연산 도중 돌발 에러가 터져도 대량 루프 전체가 다운되지 않도록 예외 처리 후 로그만 전송합니다.
        print(f"오류 발생: {d['wafer_id']} C{d['die_c']} R{d['die_r']} {d['band']} -> {e}")
        return None

# 하향 순차식 메인 실행 제어부 영역
zip_path = "./dat/HY202103"
base_save_dir = "./res/png"
target_wafers = ['D07', 'D08', 'D23', 'D24']

print("▶ 데이터 파싱을 시작합니다...")

# (주의) 상위 셀에서 선행 구동된 parse_wafer_data 함수의 리스트 변환을 실행합니다.
parsed_data_list = list(parse_wafer_data(zip_path, target_wafers))
total_items = len(parsed_data_list)

print(f"▶ 리플 제거 및 확대 플롯 생성 ({total_items}개)")

success_count = 0

# 메모리에 업로드된 카드 리스트 요소를 루프로 풀어가며 이미지 도출 함수를 반복 호출합니다.
for i, d in enumerate(parsed_data_list, 1):
    result = _draw_and_save_zoomed_plot(d, base_save_dir)

    if result is not None:
        success_count += 1

    # 장시간 구동 환경에서 상태 체크가 가능하도록 50개 단위 모니터링 카운트를 중계합니다.
    if i % 50 == 0:
        print(f"진행: {i}/{total_items}")

print(f"✅ 리플 제거 및 확대 그래프 저장 완료 (성공: {success_count}개 / 총 {total_items}개)")

▶ 데이터 파싱을 시작합니다...
▶ 리플 제거 및 확대 플롯 생성 (98개)
진행: 50/98
✅ 리플 제거 및 확대 그래프 저장 완료 (성공: 98개 / 총 98개)


### ⚡ [6] 전압에 따른 위상 변화 분석 (Phase shift - V)
* **파일명**: `Phase shift - V.py`
* **설명**: 인가된 전압($V$)에 따른 광학적 위상 변화($\Delta\phi$) 관계를 추적하여 소자의 전기-광학적 변조 효율을 분석하고, 오류나 바이어스에 따른 변화가 없는 소자들을 검출하고 저장한다.

In [7]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter

# [서버/주피터 최적화] GUI 창을 띄우지 않고 백그라운드에서 메모리 상에만 그래프를 그리는 설정입니다.
matplotlib.use("Agg")

# ── [물리 분석 하이퍼파라미터 설정] ───────────────────────────────────
# BROKEN_SLOPE_PM: 전압(V)에 따른 파장 이동량(pm)의 기울기 임계값입니다.
# 전압을 주었는데도 파장이 10 pm/V 미만으로 거의 움직이지 않는다면,
# 위상 제어 히터가 끊어졌거나 측정이 잘못된 '불량 소자(BROKEN)'로 판정합니다.
BROKEN_SLOPE_PM = 10.0

# YLIM_PM: 정상적인 소자들의 파장 이동 그래프를 한눈에 비교하기 위해 고정할 Y축 범위(pm)입니다.
# 망가진 소자는 이 거대한 범위 속에서 Y=0 라인에 납작하게 붙은 직선으로 표현되어 한눈에 불량임을 알 수 있습니다.
YLIM_PM = (-250, 550)


def _draw_and_save_phase(d, base_save_dir):
    """
    하나의 소자 데이터(d)를 입력받아 기저 광원 성분을 제거하고,
    전압별 MZM 널(Null) 파장의 이동량을 계산하여 이중 Y축 그래프(파장 변위 & 위상 변화)로 저장합니다.
    """
    try:
        # 1. REFERENCE 데이터 전처리 및 기저 신호(광원 굴곡) 모델링
        # 유효 파장 범위(wl_min ~ wl_max)에 해당하는 Reference 데이터만 True/False 마스크로 필터링합니다.
        m = (d['ref_data']['wl'] >= d['wl_min']) & (d['ref_data']['wl'] <= d['wl_max'])
        v_ref_wl = d['ref_data']['wl'][m]  # 필터링된 Reference 파장 배열
        v_ref_il = d['ref_data']['il'][m]  # 필터링된 Reference 광 강도(dB) 배열

        # 데이터 개수가 최소 31개는 되어야 연산이 가능합니다.
        if len(v_ref_wl) < 31 or np.isnan(v_ref_il).all():
            return None
        # Reference 데이터에서 자잘한 노이즈(리플)만 싹 지운 부드러운 스무딩 데이터를 생성합니다.
        sm_ref = savgol_filter(v_ref_il, 31, 3)

        # [핵심] 주피터 커널에 로드된 공용 ref_poly 함수를 호출하여 배경 광원의 3차 다항식 뼈대 함수를 얻습니다.
        poly_func = ref_poly(v_ref_wl, sm_ref, smooth=False)


        # 2. FSR(자유 스펙트럼 영역) 산출을 위한 기준 전압 데이터 선정
        # 소자의 고유 FSR을 계산하기 위해 기준이 될 -2.0V 바이어스 데이터를 찾습니다.
        # 만약 -2.0V 측정 데이터가 없다면, 리스트의 가장 첫 번째 바이어스 데이터를 대용으로 선택합니다.
        tgt_data = next(
            (b for b in d['bias_data_list'] if b['bias'] == -2.0),
            d['bias_data_list'][0] if d['bias_data_list'] else None
        )
        if tgt_data is None:
            return None

        w = tgt_data['wl']
        i = tgt_data['il']

        # 유효 분석 파장 범위 대역만 다시 필터링 마스크를 생성합니다.
        m_t = (w >= d['wl_min']) & (w <= d['wl_max'])

        if np.sum(m_t) < 31:
            return None

        # 3. 기준 데이터 평탄화 및 타겟 파장 대역의 FSR 계산
        # 기준 전압 데이터의 노이즈를 제거한 뒤, 위에서 구한 광원 기저 함수(poly_func)를 빼서 평탄화(Flattening)합니다.
        flat = savgol_filter(i[m_t], 31, 3) - poly_func(w[m_t])

        # 평탄화된 스펙트럼에서 산 모양의 봉우리(Peaks)들을 검출합니다.
        # prominence=3.0(주변보다 최소 3dB 이상 솟은 것), distance=30(피크 간 최소 간격) 조건으로 불량 피크를 거릅니다.
        peaks, _ = find_peaks(flat, prominence=3.0, distance=30)

        # 피크가 최소 2개는 검출되어야 그 사이의 간격인 FSR을 계산할 수 있습니다.
        if len(peaks) < 2:
            return None

        w_t = w[m_t]  # 유효 범위 내의 파장 값들

        # 이웃한 피크들 사이의 정중앙 지점(중심 파장)들을 계산합니다.
        centers = (w_t[peaks[:-1]] + w_t[peaks[1:]]) / 2

        # 우리가 집중적으로 분석하고자 하는 소자의 목표 관심 파장(target_wl)과 가장 가까운 중심 파장의 인덱스를 찾습니다.
        idx = np.argmin(np.abs(centers - d['target_wl']))

        if idx + 1 >= len(peaks):
            return None
        # 목표 채널의 우측 피크 파장과 좌측 피크 파장의 차이를 구하여 '진짜 FSR(nm)' 값을 확정합니다.
        FSR = w_t[peaks[idx + 1]] - w_t[peaks[idx]]

        # FSR이 0이 되어 이후 나눗셈 연산에서 ZeroDivisionError가 터지는 현상을 안전하게 방어합니다.
        if abs(FSR) < 1e-12:
            return None

        # 4. 모든 인가 전압(Bias)을 순회하며 관심 채널의 Null(최하점) 파장 추적
        phase_res = []

        for b in d['bias_data_list']:
            if b['bias'] is None:
                continue

            w_b = b['wl']
            i_b = b['il']
            m_b = (w_b >= d['wl_min']) & (w_b <= d['wl_max'])

            # 데이터 개수 미달이거나 데이터가 온통 NaN인 결측치 성분은 건너뜁니다.
            if np.sum(m_b) < 31 or np.isnan(i_b[m_b]).all():
                continue

            # [전압 데이터 평탄화] 해당 전압의 노이즈를 잡고 광원 기저 성분을 차감합니다.
            flat_b = savgol_filter(i_b[m_b], 31, 3) - poly_func(w_b[m_b])

            # 해당 전압 스펙트럼에서 똑같이 피크들을 찾아냅니다.
            p_b, _ = find_peaks(flat_b, prominence=3.0, distance=30)

            if len(p_b) < 2:
                continue

            wb_v = w_b[m_b]

            # 관심 채널 영역을 특정하기 위해 다시 피크 간 중심 파장들을 구합니다.
            cent = (wb_v[p_b[:-1]] + wb_v[p_b[1:]]) / 2

            # 목표 관심 파장과 가장 밀접한 채널의 인덱스를 추출합니다.
            i_idx = np.argmin(np.abs(cent - d['target_wl']))

            if i_idx + 1 >= len(p_b):
                continue

            # 목표 채널의 좌측 피크 인덱스(lp)와 우측 피크 인덱스(rp)를 경계선으로 정의합니다.
            lp = p_b[i_idx]
            rp = p_b[i_idx + 1]

            # [Null Valley 추적] 두 피크 사이(lp ~ rp)에서 광 세기가 가장 아래로 푹 꺼진
            v_idx = lp + np.argmin(flat_b[lp:rp + 1])

            # 전압 값과 그때 매칭되는 널 파장 값을 결과 딕셔너리에 저장합니다.
            phase_res.append({
                'bias': b['bias'],
                'v_wl': wb_v[v_idx]
            })

        if len(phase_res) == 0:
            return None

        # 5. 물리량 계산 (파장 이동량 pm 및 위상 변화량 rad 변환)
        # 전압(Bias) 순서대로 데이터를 오름차순 정렬합니다. (예: -2.0V -> -1.0V -> 0.0V ...)
        phase_res.sort(key=lambda x: x['bias'])

        # 전압이 0V에 가장 가까운 기준점(V=0) 데이터를 찾아냅니다.
        z_data = next((p for p in phase_res if abs(p['bias']) < 1e-6), None)
        if z_data is None:
            return None

        null0 = z_data['v_wl']  # 기준점이 되는 V=0 일 때의 널 파장 값입니다.

        # 넘파이 연산을 위해 리스트를 넘파이 배열(Array)형태로 변환합니다.
        biases = np.array([p['bias'] for p in phase_res])

        # Δλ (파장 변위): 현재 전압의 널 파장에서 0V 일 때의 널 파장을 빼준 뒤, pm 단위로 만들기 위해 1000을 곱합니다.(1000의 곱은 가독성을 위해)
        dlam_pm = np.array([(p['v_wl'] - null0) * 1000.0 for p in phase_res]) # 단위가 nm -> pm으로 변환

        # Δφ (위상 변화 라디안): 공식 [ 2π * (파장 이동량 / FSR) ] 을 적용해 위상 축 수치로 완전 변환합니다.
        dphi = 2 * np.pi * (dlam_pm / 1000.0) / FSR

        # 6. 불량 소자 판정 자동화 알고리즘
        # 전압 대 파장 이동량(dlam_pm) 데이터를 1차 직선($y=ax+b$)으로 피팅하여 기울기($a$)를 구합니다.
        slope_pm = np.polyfit(biases, dlam_pm, 1)[0]  # 단위: pm/V

        # 구한 기울기의 절대값이 사용자가 설정한 임계값(10 pm/V)보다 작으면 망가진 소자로 판정합니다.
        is_broken = abs(slope_pm) < BROKEN_SLOPE_PM

        # 망가진 소자라면 파일명 뒤에 자동으로 구별용 태그(_BROKEN)를 붙여줍니다.
        tag = "_BROKEN" if is_broken else ""

        # 7. 이중 Y축 고급 그래프 시각화 작도 (왼쪽: 파장 이동, 오른쪽: 위상)

        # 도화지와 첫 번째 왼쪽 축(ax1) 인스턴스를 생성합니다.
        fig, ax1 = plt.subplots(figsize=(10, 6))

        # [왼쪽 Y축 - 파장 이동 플롯] 파란색 톤의 선과 채워진 원형 마커로 드로잉합니다.
        ax1.plot(biases, dlam_pm, 'o-', color='steelblue', lw=2.5, ms=8, label='Δλ_null (pm)')

        # 그래프 해독을 도와줄 수평/수직 x=0, y=0 기준 가이드 가로선을 배치합니다.
        ax1.axhline(0, color='gray', ls='--', lw=2, alpha=0.6)
        ax1.axvline(0, color='gray', ls='--', lw=2, alpha=0.6)

        # 축 라벨을 굵고 선명하게 명시합니다.
        ax1.set_xlabel("Bias Voltage (V)", fontsize=16, fontweight='bold')
        ax1.set_ylabel("Δλ_null vs V=0 [pm]", fontsize=16, fontweight='bold', color='steelblue')
        ax1.tick_params(axis='y', labelcolor='steelblue')  # 좌측 눈금 글자 색상을 플롯과 통일합니다.

        ax1.set_ylim(min(YLIM_PM[0], dlam_pm.min() - 20), max(YLIM_PM[1], dlam_pm.max() + 20))
        ax1.grid(True, ls='--', alpha=0.6, lw=1) # 투명한 점선 그리드를 격자로 그리게 합니다.

        ax2 = ax1.twinx()

        ax2.plot(biases, dphi, 's--', color='crimson', lw=1.5, ms=6, label='Δφ (rad)')

        ax2.axhline(np.pi, color='crimson', ls=':', lw=1, alpha=0.5)
        ax2.text(biases.max(), np.pi, ' π', color='crimson', va='center', fontsize=10)
        ax2.set_ylabel("Δφ [rad] (= 2π·Δλ/FSR)", fontsize=16, fontweight='bold', color='crimson')
        ax2.tick_params(axis='y', labelcolor='crimson') # 우측 눈금 글자 색상을 붉은색으로 일치시킵니다.

        band_label = 'C' if 'LMZC' in d['band'] else 'O'
        broken_txt = "  [BROKEN: dλ/dV≈0]" if is_broken else ""
        ax1.set_title(
            f"Wafer: {d['wafer_id']} / Coord: ({d['die_c']}, {d['die_r']}) / {band_label}-band{broken_txt}\n"
            f"Phase Shift  |  dλ/dV = {slope_pm:.1f} pm/V ; FSR = {FSR:.2f} nm",
            fontsize=15, fontweight='bold', pad=15, color='black'
        )

        for spine in ax1.spines.values():
            spine.set_linewidth(2)

        fig.tight_layout()

        date_str = d.get('date', 'Unknown_Date')
        save_dir = os.path.join(base_save_dir, d['wafer_id'], date_str)
        os.makedirs(save_dir, exist_ok=True)  # 폴더가 없으면 에러 없이 자동 생성합니다.

        # 최종 저장 파일명을 조립합니다. (예: D07_C1_R2_LMZC_Phase_BROKEN.png)
        filename = f"{d['wafer_id']}_C{d['die_c']}_R{d['die_r']}_{d['band']}_Phase{tag}.png"

        fig.savefig(os.path.join(save_dir, filename), bbox_inches='tight', dpi=150)
        plt.close(fig)  # 사용이 완료된 캔버스를 메모리에서 즉시 해제(파괴)합니다.

        return filename

    except Exception as e:
        # 특정 다이 데이터 처리 중 돌발 에러가 터져도 전체 98개 루프가 다운되지 않도록 예외 차단 후 로그를 기록합니다.
        print(f"오류 발생: {d['wafer_id']} C{d['die_c']} R{d['die_r']} {d['band']} -> {e}")
        return None


# ── [주피터 메인 실행부 제어 영역] ────────────────────────────────────
zip_path = "./dat/HY202103"
base_save_dir = "./res/png"
target_wafers = ['D07', 'D08', 'D23', 'D24']

print("▶ 데이터 파싱을 시작합니다...")
# (주의) 선행 셀에 이미 정의되어 구동된 parse_wafer_data 데이터 생성기를 리스트로 풀어서 로드합니다.
parsed_data_list = list(parse_wafer_data(zip_path, target_wafers))
total_items = len(parsed_data_list)

print(f"▶ Phase Shift (dλ/dV) 플롯 생성 ({total_items}개)")

success_count = 0

# 주피터 커널 환경에 가장 안전한 1스레드 순차 제어 루프로 처리를 개시합니다.
for i, d in enumerate(parsed_data_list, 1):
    result = _draw_and_save_phase(d, base_save_dir)

    if result is not None:
        success_count += 1

    # 장시간 연산 작업 시 진행 상황을 파악할 수 있도록 50개 단위로 마일스톤 로그를 중계합니다.
    if i % 50 == 0:
        print(f"진행: {i}/{total_items}")

print(f"✅ Phase Shift 저장 완료 (성공: {success_count}개 / 총 {total_items}개)")

▶ 데이터 파싱을 시작합니다...
▶ Phase Shift (dλ/dV) 플롯 생성 (98개)
진행: 50/98
✅ Phase Shift 저장 완료 (성공: 98개 / 총 98개)


### 🔋 [7] VpiL 산출 (VpiL Calculation)
* **파일명**: `VpiL.py`
* **설명**: 반파장 전압(Vpi)과 소자 길이(L)의 곱인 $V_\pi\cdot L$을 계산하여 단위 길이당 변조 효율 지표를 도출합니다.

In [8]:
import os
from scipy.signal import find_peaks, savgol_filter

def _process_vpil(d, base_res_dir, L_length):
    """
    [개별 Die별 VpiL 곡선 시각화 및 저장 함수]
    - 특정 Die의 전압별 데이터를 분석하여 VpiL 곡선을 그리고, 0V에서의 효율을 마킹하여 이미지로 저장합니다.
    """
    try:
        # ---------------------------------------------------------
        # 기본 메타데이터 추출 및 파장 영역 필터링
        # ---------------------------------------------------------
        wafer = d['wafer_id']
        band = d['band']
        c = d['die_c']
        r = d['die_r']
        date_str = d.get('date', 'Unknown_Date')

        # 분석을 진행할 유효 파장 대역(wl_min ~ wl_max)의 마스크 생성
        m = (d['ref_data']['wl'] >= d['wl_min']) & (d['ref_data']['wl'] <= d['wl_max'])
        v_ref_wl = d['ref_data']['wl'][m]
        v_ref_il = d['ref_data']['il'][m]

        # 데이터 개수가 너무 적거나 전부 NaN(결측치)이면 정상적인 분석이 불가능하므로 스킵
        if len(v_ref_wl) < 31:
            return None
        if np.isnan(v_ref_il).all():
            return None

        # ---------------------------------------------------------
        # Reference 데이터 평탄화 및 Baseline 다항식 피팅
        # ---------------------------------------------------------
        # Savitzky-Golay 필터로 레퍼런스의 고주파 노이즈를 1차 제거
        sm_ref = savgol_filter(v_ref_il, 31, 3)
        # 광섬유 결합 손실 등 거시적인 기저선(Baseline)을 제거하기 위해 3차 다항식으로 피팅
        poly_func = np.poly1d(np.polyfit(v_ref_wl, sm_ref, 3))

        # ---------------------------------------------------------
        # 0V (Zero Bias) 기준 데이터 추출 및 초기 파장/FSR 분석
        # ---------------------------------------------------------
        # Bias 전압이 정확히 0.0V인 데이터를 기준점으로 찾음
        z_data = next((b for b in d['bias_data_list'] if b['bias'] == 0.0), None)
        if z_data is None:
            return None

        # 0V 데이터 유효 대역 필터링
        m0 = (z_data['wl'] >= d['wl_min']) & (z_data['wl'] <= d['wl_max'])
        w0 = z_data['wl'][m0]
        i0 = z_data['il'][m0]

        if len(w0) < 31:
            return None

        # 0V 투과 스펙트럼 노이즈 제거 후, 레퍼런스 배경(poly_func)을 빼서 pure fringe 신호만 남김
        flat0 = savgol_filter(i0, 31, 3) - poly_func(w0)

        # 상쇄 간섭 지점(골짜기)을 찾기 위해 신호를 뒤집어(-flat0) 피크 검출 진행
        v0, _ = find_peaks(-flat0, prominence=0.3, distance=20)

        if len(v0) < 2:
            return None  # 주기를 계산하려면 최소 2개 이상의 골짜기가 필요함

        # FSR (자유 스펙트럼 범위): 골짜기 간 간격의 평균값
        fsr = np.mean(np.diff(w0[v0]))
        if abs(fsr) < 1e-12:
            return None  # 영(0)으로 나누는 오류 방지

        # 실험 타겟 파장(target_wl)에 가장 인접한 골짜기 하나를 '추적 중심 파장(cwl)'으로 고정
        cwl = w0[v0[np.argmin(np.abs(w0[v0] - d['target_wl']))]]

        # ---------------------------------------------------------
        # 전압(Bias) 인가에 따른 중심 파장 이동(Phase Shift) 추적
        # ---------------------------------------------------------
        volts = []
        p_pi = []
        sh = fsr / 2.5  # 인접 주기로 튀는 것을 막기 위한 국소 탐색 윈도우 (FSR의 40% 크기)

        for b in d['bias_data_list']:
            if b['bias'] is None:
                continue

            w = b['wl']
            i = b['il']

            mb = (w >= d['wl_min']) & (w <= d['wl_max'])
            wb = w[mb]
            ib = i[mb]

            # 0V 중심 파장(cwl) 근방의 범위만 마스킹
            mloc = (wb >= cwl - sh) & (wb <= cwl + sh)

            # Savitzky-Golay 필터의 최소 요구 조건(윈도우 크기=11) 충족 여부 확인
            if np.sum(mloc) < 11:
                continue

            # 국소 영역 평탄화 진행
            flat = savgol_filter(ib[mloc], 11, 3) - poly_func(wb[mloc])

            # 커스텀 포물선 서브피팅 함수(q_sub)로 정밀 소수점 단위 골짜기 파장(vwl) 도출
            vwl = q_sub(wb[mloc], flat)

            volts.append(b['bias'])
            # 이동 거리를 기반으로 위상 변화량(Phase Shift) 계산 후 적재
            p_pi.append(2.0 * (vwl - cwl) / fsr)

        if len(volts) < 5:
            return None  # 신뢰성 있는 2차 곡선 피팅을 위한 최소 전압 포인트 수 검증

        volts = np.array(volts)
        p_pi = np.array(p_pi)

        # ---------------------------------------------------------
        # VpiL 계산 및 미분 기울기 도출
        # ---------------------------------------------------------
        # 전압-위상 데이터를 2차 다항식으로 피팅
        fit = np.poly1d(np.polyfit(volts, p_pi, 2))
        # 도함수(미분) 구하기
        derivative = np.polyder(fit)

        # 0V 순간의 접선 기울기 최댓값 처리 (분모가 0이 되는 현상 방지)
        slope_at_0 = max(abs(derivative(0.0)), 1e-5)
        # 0V에서의 VpiL 효율 계산
        vpil_0V = L_length / slope_at_0

        # 물리적인 상식선을 벗어나는 소자는 비정상으로 간주하고 필터링
        if not (0.1 <= vpil_0V <= 10.0):
            return None

        # ---------------------------------------------------------
        # Matplotlib을 이용한 개별 Die 분석 그래프 플로팅
        # ---------------------------------------------------------
        fig, ax = plt.subplots(figsize=(10, 6))

        # 전압 연속 변화에 따른 VpiL 커브 계산
        vpil_curve = L_length / np.maximum(np.abs(derivative(volts)), 1e-5)

        # 전체 VpiL 경향성 곡선 그리기
        ax.plot(volts, vpil_curve, 's-', linewidth=2.5, markersize=8, color='gray', label='VpiL Curve')

        # 핵심이 되는 0V 지점 효율을 빨간색 별표(*)로 강조 마킹
        ax.plot(0.0, vpil_0V, 'r*', markersize=18, label=f'Value @ 0V = {vpil_0V:.3f}')

        # 시인성을 높이기 위한 가이드 점선 배치
        ax.axhline(vpil_0V, color='red', linestyle=':', linewidth=2, alpha=0.6)
        ax.axvline(0.0, color='red', linestyle=':', linewidth=2, alpha=0.6)

        # 타이틀 및 축 레이블 굵게 스타일링
        ax.set_title(f"Wafer: {wafer} / Coord: ({c}, {r}) / Band: {band}\nVparentL at 0V", fontsize=18, fontweight='bold', pad=15)
        ax.set_xlabel("Voltage (V)", fontsize=16, fontweight='bold')
        ax.set_ylabel("Vpi*L (V*cm)", fontsize=16, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=0.6, linewidth=1)

        # 범례 안전하게 노출 처리
        handles, labels = ax.get_legend_handles_labels()
        if len(handles) > 0:
            ax.legend(loc='best', prop={'size': 12, 'weight': 'bold'})

        # 차트 외곽 테두리선 두께 조절
        for spine in ax.spines.values():
            spine.set_linewidth(2)

        fig.tight_layout()

        # ---------------------------------------------------------
        # 이미지 파일 디렉토리 자동 생성 및 저장
        # ---------------------------------------------------------
        save_dir = os.path.join(base_res_dir, wafer, date_str)
        os.makedirs(save_dir, exist_ok=True)

        filename = f"{wafer}_C{c}_R{r}_{band}_VpiL.png"
        fig.savefig(os.path.join(save_dir, filename), bbox_inches='tight', dpi=150)
        plt.close(fig)  # 메모리 누수 방지를 위한 피겨 닫기

        return True

    except Exception as e:
        print(f"오류 발생: {wafer} C{c} R{r} {band} -> {e}")
        return None

# ==========================================================
# 메인 실행부
# ==========================================================
zip_path = "./dat/HY202103"
base_res_dir = "./res/png"
target_wafers = ['D07', 'D08', 'D23', 'D24']
L_length = 0.05

os.makedirs(base_res_dir, exist_ok=True)
print("🚀 0V 기준 개별 다이 그래프 생성을 시작합니다...")

# 파서 엔진을 구동해 원본 데이터를 리스트화
parsed_data_list = list(parse_wafer_data(zip_path, target_wafers))

valid_count = 0
total_items = len(parsed_data_list)

# 순차적으로 개별 다이를 순회하며 연산 및 이미지 파일 생성 진행
for i, d in enumerate(parsed_data_list, 1):
    result = _process_vpil(d, base_res_dir, L_length)

    if result:
        valid_count += 1

    # 50개 주기마다 콘솔창에 현재 진행 상황 플래그 출력
    if i % 50 == 0:
        print(f"진행: {i}/{total_items}")

print(f"✅ 개별 그래프 저장 완료 (유효 Die: {valid_count}개)")

🚀 0V 기준 개별 다이 그래프 생성을 시작합니다...
진행: 50/98
✅ 개별 그래프 저장 완료 (유효 Die: 73개)


### 🌓 [8] 소멸비 분석 (Extinction Ratio Analysis)
* **파일명**: `ER_Analysis.py`
* **설명**: 최대 투과 조건(Max)과 최소 투과 조건(Min)의 차이인 소멸비(Extinction Ratio, ER)를 산출하여 소자의 온/오프 효율을 평가합니다.

In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

zip_path = "./dat/HY202103"
base_res_dir = "./res/png"
target_wafers = ['D07', 'D08', 'D23', 'D24']

print("🚀 평탄화 로직 기반 ER 추출 및 날짜별/Center vs Edge 분석을 시작합니다...")

# 1. WaferMap과 BoxPlot 최상위 경로 설정
wafer_map_dir = os.path.join(base_res_dir, "WaferMap")
box_plot_dir = os.path.join(base_res_dir, "BoxPlot")
os.makedirs(wafer_map_dir, exist_ok=True)
os.makedirs(box_plot_dir, exist_ok=True)

er_data_list = []
count = 0

for d in parse_wafer_data(zip_path, target_wafers):
    date_str = d.get('date', 'Unknown_Date')

    m = (d['ref_data']['wl'] >= d['wl_min']) & (d['ref_data']['wl'] <= d['wl_max'])
    v_ref_wl, v_ref_il = d['ref_data']['wl'][m], d['ref_data']['il'][m]
    if len(v_ref_wl) < 4: continue

    poly_func = np.poly1d(np.polyfit(v_ref_wl, v_ref_il, 3))
    max_er = 0.0

    for b in d['bias_data_list']:
        m_b = (b['wl'] >= d['wl_min']) & (b['wl'] <= d['wl_max'])
        v_wl, v_il = b['wl'][m_b], b['il'][m_b]
        if len(v_wl) == 0: continue

        flat_il = v_il - poly_func(v_wl)
        peaks, _ = find_peaks(flat_il, prominence=3.0, distance=30)

        if len(peaks) >= 2:
            trend = np.poly1d(np.polyfit(v_wl[peaks], flat_il[peaks], 1))
            final_il = flat_il - trend(v_wl)
        else:
            final_il = flat_il

        er = np.percentile(final_il, 99) - np.percentile(final_il, 1)
        max_er = max(max_er, er)

    er_data_list.append({
        'Wafer': d['wafer_id'], 'Band': d['band'], 'Date': date_str, 'Column': d['die_c'], 'Row': d['die_r'],
        'Radius': np.sqrt(d['die_c'] ** 2 + d['die_r'] ** 2), 'ER': max_er
    })

    count += 1
    if count % 100 == 0: print(f"현재 {count}개 Die 파싱 및 평탄화 ER 추출 완료...")

df = pd.DataFrame(er_data_list)


# 2. 자정을 넘긴 측정 데이터(0603, 0604) 병합 처리

def merge_midnight_dates(date_val):
    date_str = str(date_val)
    if '0603' in date_str or '0604' in date_str:
        return '20190603'
    return date_str

df['Date'] = df['Date'].apply(merge_midnight_dates)
print("🕒 0603 및 0604 날짜 데이터를 '0603_0604_Combined'로 통합했습니다.")

filtered_df = pd.DataFrame()

for (wafer, band, date), group in df.groupby(['Wafer', 'Band', 'Date']):
    if len(group) > 5:
        m_val, s_val = group['ER'].mean(), group['ER'].std()
        filtered_df = pd.concat(
            [filtered_df, group[(group['ER'] >= m_val - 3 * s_val) & (group['ER'] <= m_val + 3 * s_val)]])
    else:
        filtered_df = pd.concat([filtered_df, group])

max_rad = filtered_df['Radius'].max()
edge_thresh = max_rad * 0.75
filtered_df['Region'] = np.where(filtered_df['Radius'] > edge_thresh, 'Edge', 'Center')

# [통일된 디자인] Wafer Map 그리기
print("▶ 웨이퍼 및 날짜별 Wafer Map 생성 중...")

# 격자 범위를 결정하기 위해 물리적 반경(`max_rad`)을 기준으로 축 Limits 설정 (대칭)
map_limit = np.ceil(max_rad) + 0.5

for b in filtered_df['Band'].unique():
    band_limits = {'min': np.floor(filtered_df[filtered_df['Band'] == b]['ER'].min()),
                   'max': np.ceil(filtered_df[filtered_df['Band'] == b]['ER'].max())}

    for (w, date), group in filtered_df[filtered_df['Band'] == b].groupby(['Wafer', 'Date']):
        if group.empty: continue

        # figsize를 정방형으로 유지
        plt.figure(figsize=(10, 10))
        ax = plt.gca()  # 현재 Axes 객체 가져오기

        # 1. 웨이퍼 외곽선 및 Edge 영역 구분선 (zorder=1: 맨 아래)
        theta = np.linspace(0, 2 * np.pi, 100)
        ax.plot((max_rad + 0.5) * np.cos(theta), (max_rad + 0.5) * np.sin(theta), color='#555555', lw=2, zorder=1)
        ax.plot(edge_thresh * np.cos(theta), edge_thresh * np.sin(theta), color='#FF8888', ls='--', lw=2, alpha=0.7,
                zorder=1)

        # 2. 바둑판 격자 및 축 설정
        ax.set_aspect('equal')  # 정방형 비율 유지

        # 축 범위 설정 (0,0 기준 대칭)
        ax.set_xlim(-map_limit, map_limit)
        ax.set_ylim(-map_limit, map_limit)

        # Ticks 설정: 정수 좌표에 눈금을 매김
        ticks = np.arange(-np.floor(map_limit), np.ceil(map_limit) + 1, 1)
        ax.set_xticks(ticks)
        ax.set_yticks(ticks)

        # 격자 추가 (zorder=2: 외곽선 위, 데이터 아래)
        ax.grid(True, which='major', axis='both', color='#DDDDDD', linestyle='--', linewidth=1, zorder=2)

        # 축 라벨 스타일 및 표시 설정
        ax.tick_params(axis='both', which='major', labelsize=11, labelcolor='#333333')
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontweight('bold')

        # 축 이름 설정
        ax.set_xlabel("Column (X Coordinate)", fontsize=14, fontweight='bold', labelpad=10)
        ax.set_ylabel("Row (Y Coordinate)", fontsize=14, fontweight='bold', labelpad=10)

        # 3. 데이터 포인트 플로팅 (zorder=5: 격자 위)
        # ER 특성에 맞게 cmap='coolwarm_r' 유지
        sc = ax.scatter(group['Column'], group['Row'], c=group['ER'], cmap='coolwarm_r',
                        vmin=band_limits['min'], vmax=band_limits['max'], s=500, edgecolor='black', alpha=0.9,
                        zorder=5)

        # 4. 데이터 값 텍스트 표시 (zorder=6: 데이터 포인트 위)
        for _, row in group.iterrows():
            ax.text(row['Column'], row['Row'], f"{row['ER']:.1f}", ha='center', va='center', fontsize=9,
                    color='black', fontweight='bold', zorder=6)

        # 컬러바 설정
        cb = plt.colorbar(sc, shrink=0.8, pad=0.03)
        cb.set_label('Extinction Ratio [dB]', fontsize=13, fontweight='bold')
        cb.ax.tick_params(labelsize=11)
        for l in cb.ax.yaxis.get_ticklabels(): l.set_weight("bold")

        # 제목 설정
        plt.title(f"Wafer Map: {w} / {b} (Flattened ER)\nDate: {date}", fontsize=17, fontweight='bold', pad=20)

        # 경로: WaferMap / 웨이퍼 / 날짜
        w_dir = os.path.join(wafer_map_dir, w, date)
        os.makedirs(w_dir, exist_ok=True)
        # bbox_inches='tight'로 축 라벨이 그림 밖으로 잘 나가지 않게 보호
        plt.savefig(os.path.join(w_dir, f"Map_{w}_{b}_{date}_ER.png"), bbox_inches='tight', dpi=100)
        plt.close()
# 3. [통일된 디자인] Box Plot 그리기 (개별 웨이퍼 단위)

print("▶ 웨이퍼 및 날짜별 Box Plot 생성 중...")
for (w, b, date), wbd_df in filtered_df.groupby(['Wafer', 'Band', 'Date']):
    if wbd_df.empty: continue

    plt.figure(figsize=(8, 8))  # 개별 웨이퍼 플롯 사이즈
    pos, data, labels, colors = [], [], [], []

    c = wbd_df[wbd_df['Region'] == 'Center']['ER'].values
    e = wbd_df[wbd_df['Region'] == 'Edge']['ER'].values

    if len(c) > 0:
        pos.append(1)
        data.append(c)
        labels.append(f"Center\nn={len(c)}")
        colors.append('#3498db')
    if len(e) > 0:
        pos.append(2)
        data.append(e)
        labels.append(f"Edge\nn={len(e)}")
        colors.append('#e74c3c')

    if not data: continue

    # Y축 범위를 미리 계산하여 배경색 영역 지정에 사용
    all_data = np.concatenate(data)
    y_min = min(all_data.min(), 15.0) - 2  # 데이터 최소값 또는 15dB 중 작은 값 기준 안전마진
    y_max = max(all_data.max(), 25.0) + 2  # 데이터 최대값 또는 25dB 중 큰 값 기준 안전마진
    plt.ylim(y_min, y_max)

    # 🌟 성능별 배경색(수평 영역) 지정

    # 타겟(20.0) 이상: 연한 초록색 (Good)
    plt.axhspan(20.0, y_max, facecolor='#e8f8f5', alpha=0.6, zorder=0, label='Good Region')

    # 타겟(20.0) 미만: 연한 붉은/코랄 계열 (Poor) - 눈에 띄면서 부정적인 의미 전달
    plt.axhspan(y_min, 20.0, facecolor='#fadbd8', alpha=0.6, zorder=0, label='Poor Region')


    box = plt.boxplot(data, positions=pos, patch_artist=True, widths=0.5,
                      flierprops=dict(marker='d', markerfacecolor='black', markersize=6, alpha=0.6),
                      zorder=2)
    for p, c_hex in zip(box['boxes'], colors): p.set_facecolor(c_hex); p.set_alpha(0.7)

    # Jitter
    for p, d_arr in zip(pos, data):
        plt.scatter(np.random.normal(p, 0.05, len(d_arr)), d_arr, color='black', alpha=0.5, s=20, zorder=3)

    # 평균 및 타겟 라인
    avg_er = wbd_df['ER'].mean()
    plt.axhline(avg_er, color='blue', ls='--', lw=2.5, label=f'Avg: {avg_er:.2f} dB', zorder=4)
    plt.axhline(20.0, color='red', ls='-', lw=2.5, label='Target: 20.00 dB', zorder=4)

    plt.title(f"ER Analysis : {w} ({b})\nDate: {date}", fontsize=18, fontweight='bold', pad=15)
    plt.xticks(pos, labels, fontsize=14, fontweight='bold')
    plt.yticks(fontsize=14, fontweight='bold')
    plt.ylabel('Extinction Ratio [dB]', fontsize=16, fontweight='bold')

    plt.legend(loc='upper right', prop={'size': 11, 'weight': 'bold'})
    plt.grid(True, axis='y', ls=':', alpha=0.4, zorder=1)
    plt.xlim(0.5, max(pos) + 0.5)
    plt.tight_layout()

    # 경로: BoxPlot / 웨이퍼 / 날짜
    box_dir = os.path.join(box_plot_dir, w, date)
    os.makedirs(box_dir, exist_ok=True)
    plt.savefig(os.path.join(box_dir, f"Box_{w}_{b}_{date}_ER_Flattened.png"), bbox_inches='tight')
    plt.close()

🚀 평탄화 로직 기반 ER 추출 및 날짜별/Center vs Edge 분석을 시작합니다...
🕒 0603 및 0604 날짜 데이터를 '0603_0604_Combined'로 통합했습니다.
▶ 웨이퍼 및 날짜별 Wafer Map 생성 중...
▶ 웨이퍼 및 날짜별 Box Plot 생성 중...


### 📉 [9] 삽입 손실 분석 (Insertion Loss Analysis)
* **파일명**: `IL_Analysis.py`
* **설명**: MZM 소자가 통과시키는 최대 광출력을 기반으로 삽입 손실(Insertion Loss, IL) 값을 계산하고 통계치를 추출합니다.

In [10]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

zip_path = "./dat/HY202103"
base_res_dir = "./res/png"
target_wafers = ['D07', 'D08', 'D23', 'D24']
IL_TARGETS = {'LMZO': -8.75, 'LMZC': -8.75}

# 1. WaferMap과 BoxPlot 최상위 경로 설정
wafer_map_dir = os.path.join(base_res_dir, "WaferMap")
box_plot_dir = os.path.join(base_res_dir, "BoxPlot")
os.makedirs(wafer_map_dir, exist_ok=True)
os.makedirs(box_plot_dir, exist_ok=True)

print("🚀 IL 추출 및 날짜별/Center vs Edge 통합 분석을 시작합니다...")

il_data_list = []
for d in parse_wafer_data(zip_path, target_wafers):
    date_str = d.get('date', 'Unknown_Date')

    max_peak = -999.0
    for b in d['bias_data_list']:
        if b['bias'] is None: continue
        m = (b['wl'] >= d['wl_min']) & (b['wl'] <= d['wl_max'])
        if np.any(m): max_peak = max(max_peak, np.max(b['il'][m]))

    if max_peak != -999.0:
        il_data_list.append(
            {'Wafer': d['wafer_id'], 'Band': d['band'], 'Date': date_str, 'Column': d['die_c'], 'Row': d['die_r'],
             'IL': max_peak})

df = pd.DataFrame(il_data_list).groupby(['Wafer', 'Band', 'Date', 'Column', 'Row'], as_index=False)['IL'].mean()
df['Radius'] = np.sqrt(df['Column'] ** 2 + df['Row'] ** 2)


# 2. 자정을 넘긴 측정 데이터(0603, 0604) 병합 처리

def merge_midnight_dates(date_val):
    date_str = str(date_val)
    if '0603' in date_str or '0604' in date_str:
        return '20190603'
    return date_str


df['Date'] = df['Date'].apply(merge_midnight_dates)
print("🕒 0603 및 0604 날짜 데이터를 '0603_0604_Combined'로 통합했습니다.")


filtered_df = pd.DataFrame()
for _, group in df.groupby(['Wafer', 'Band', 'Date']):
    m, s = group['IL'].mean(), group['IL'].std()
    filtered_df = pd.concat([filtered_df, group[(group['IL'] >= m - 3 * s) & (group['IL'] <= m + 3 * s)]])

max_rad = filtered_df['Radius'].max()
filtered_df['Region'] = np.where(filtered_df['Radius'] > max_rad * 0.75, 'Edge', 'Center')
# ==========================================================
# [통일된 디자인] Wafer Map 그리기 (격자 및 좌표 추가 수정본)
# ==========================================================
print("▶ 웨이퍼 및 날짜별 Wafer Map 생성 중...")

# 격자 범위를 결정하기 위해 물리적 반경(`max_rad`)을 기준으로 축 Limits 설정 (대칭)
map_limit = np.ceil(max_rad) + 0.5

for b in filtered_df['Band'].unique():
    limits = {'min': np.floor(filtered_df[filtered_df['Band'] == b]['IL'].min()),
              'max': np.ceil(filtered_df[filtered_df['Band'] == b]['IL'].max())}

    for (w, date), grp in filtered_df[filtered_df['Band'] == b].groupby(['Wafer', 'Date']):
        if grp.empty: continue

        # figsize를 정방형으로 유지
        plt.figure(figsize=(10, 10))
        ax = plt.gca()  # 현재 Axes 객체 가져오기

        # 1. 웨이퍼 외곽선 및 Edge 영역 구분선 (zorder=1: 맨 아래)
        th = np.linspace(0, 2 * np.pi, 100)
        ax.plot((max_rad + 0.5) * np.cos(th), (max_rad + 0.5) * np.sin(th), color='#555555', lw=2, zorder=1)
        ax.plot(max_rad * 0.75 * np.cos(th), max_rad * 0.75 * np.sin(th), color='#FF8888', ls='--', lw=2, alpha=0.7,
                zorder=1)

        # 2. 바둑판 격자 및 축 설정
        ax.set_aspect('equal')  # 정방형 비율 유지

        # 축 범위 설정 (0,0 기준 대칭)
        ax.set_xlim(-map_limit, map_limit)
        ax.set_ylim(-map_limit, map_limit)

        # Ticks 설정: 정수 좌표에 눈금을 매김
        ticks = np.arange(-np.floor(map_limit), np.ceil(map_limit) + 1, 1)
        ax.set_xticks(ticks)
        ax.set_yticks(ticks)

        # 격자 추가 (zorder=2: 외곽선 위, 데이터 아래)
        ax.grid(True, which='major', axis='both', color='#DDDDDD', linestyle='--', linewidth=1, zorder=2)

        # 축 라벨 스타일 및 표시 설정
        ax.tick_params(axis='both', which='major', labelsize=11, labelcolor='#333333')
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontweight('bold')

        # 축 이름 설정
        ax.set_xlabel("Column (X Coordinate)", fontsize=14, fontweight='bold', labelpad=10)
        ax.set_ylabel("Row (Y Coordinate)", fontsize=14, fontweight='bold', labelpad=10)

        # 3. 데이터 포인트 플로팅 (zorder=5: 격자 위)
        sc = ax.scatter(grp['Column'], grp['Row'], c=grp['IL'], cmap='coolwarm_r', vmin=limits['min'],
                        vmax=limits['max'], s=500, edgecolor='black', alpha=0.9, zorder=5)

        # 4. 데이터 값 텍스트 표시 (zorder=6: 데이터 포인트 위)
        for _, r in grp.iterrows():
            ax.text(r['Column'], r['Row'], f"{r['IL']:.1f}", ha='center', va='center',
                    fontsize=9, fontweight='bold', color='black', zorder=6)

        # 컬러바 설정
        cb = plt.colorbar(sc, shrink=0.8, pad=0.03)
        cb.set_label('IL [dB]', fontsize=13, fontweight='bold')
        cb.ax.tick_params(labelsize=11)
        for l in cb.ax.yaxis.get_ticklabels(): l.set_weight("bold")

        # 제목 설정
        plt.title(f"Wafer Map: {w} / {b} (IL)\nDate: {date}", fontsize=17, fontweight='bold', pad=20)

        # 저장 경로: WaferMap / 웨이퍼 / 날짜
        w_dir = os.path.join(wafer_map_dir, w, date)
        os.makedirs(w_dir, exist_ok=True)
        plt.savefig(os.path.join(w_dir, f"Map_{w}_{b}_{date}_IL.png"), bbox_inches='tight', dpi=100)
        plt.close()

# 3. [통일된 디자인] Box Plot 그리기

print("▶ 웨이퍼 및 날짜별 Box Plot 생성 중...")
for (w, b, date), wbd_df in filtered_df.groupby(['Wafer', 'Band', 'Date']):
    if wbd_df.empty: continue

    plt.figure(figsize=(8, 8))  # 개별 웨이퍼 플롯 사이즈
    pos, data, lbl, clr = [], [], [], []

    c = wbd_df[wbd_df['Region'] == 'Center']['IL'].values
    e = wbd_df[wbd_df['Region'] == 'Edge']['IL'].values

    if len(c) > 0:
        pos.append(1)
        data.append(c)
        lbl.append(f"Center\nn={len(c)}")
        clr.append('#3498db')
    if len(e) > 0:
        pos.append(2)
        data.append(e)
        lbl.append(f"Edge\nn={len(e)}")
        clr.append('#e74c3c')

    if not data: continue

    tgt_il = IL_TARGETS.get(b, -8.75)

    # Y축 범위를 미리 계산하여 배경색 영역 지정에 사용
    all_data = np.concatenate(data)
    y_min = min(all_data.min(), tgt_il) - 2  # 데이터 최소값 또는 타겟 중 작은 값 기준 안전마진
    y_max = max(all_data.max(), tgt_il) + 2  # 데이터 최대값 또는 타겟 중 큰 값 기준 안전마진
    plt.ylim(y_min, y_max)

    # ------------------------------------------------------
    # 🌟 성능별 배경색(수평 영역) 지정
    # ------------------------------------------------------
    # 타겟(Target) 이상: 연한 초록색 (Good Region)
    plt.axhspan(tgt_il, y_max, facecolor='#e8f8f5', alpha=0.6, zorder=0, label='Good Region')

    # 타겟(Target) 미만: 연한 붉은/코랄 계열 (Poor Region) - 눈에 띄면서 부정적인 의미 전달
    plt.axhspan(y_min, tgt_il, facecolor='#fadbd8', alpha=0.6, zorder=0, label='Poor Region')
    # ------------------------------------------------------

    box = plt.boxplot(data, positions=pos, patch_artist=True, widths=0.5,
                      flierprops=dict(marker='d', markerfacecolor='black', markersize=6, alpha=0.6),
                      zorder=2)
    for p, c_hex in zip(box['boxes'], clr): p.set_facecolor(c_hex); p.set_alpha(0.7)

    # Jitter(산점도) 추가
    for p, d_arr in zip(pos, data):
        plt.scatter(np.random.normal(p, 0.05, len(d_arr)), d_arr, color='black', alpha=0.5, s=20, zorder=3)

    avg_il = wbd_df['IL'].mean()

    # 평균 및 타겟 라인
    plt.axhline(avg_il, color='blue', ls='--', lw=2.5, label=f'Avg: {avg_il:.2f} dB', zorder=4)
    plt.axhline(tgt_il, color='red', ls='-', lw=2.5, label=f'Target: {tgt_il:.2f} dB', zorder=4)

    plt.title(f"IL Analysis : {w} ({b})\nDate: {date}", fontsize=18, fontweight='bold', pad=15)
    plt.xticks(pos, lbl, fontsize=14, fontweight='bold')
    plt.yticks(fontsize=14, fontweight='bold')
    plt.ylabel('IL [dB]', fontsize=16, fontweight='bold')

    # 범례 설정
    plt.legend(loc='upper right', prop={'size': 11, 'weight': 'bold'})
    plt.grid(True, axis='y', ls=':', alpha=0.4, zorder=1)
    plt.xlim(0.5, max(pos) + 0.5)
    plt.tight_layout()

    # 저장 경로: BoxPlot / 웨이퍼 / 날짜
    box_dir = os.path.join(box_plot_dir, w, date)
    os.makedirs(box_dir, exist_ok=True)
    plt.savefig(os.path.join(box_dir, f"Box_{w}_{b}_{date}_IL.png"), bbox_inches='tight')
    plt.close()

🚀 IL 추출 및 날짜별/Center vs Edge 통합 분석을 시작합니다...
🕒 0603 및 0604 날짜 데이터를 '0603_0604_Combined'로 통합했습니다.
▶ 웨이퍼 및 날짜별 Wafer Map 생성 중...
▶ 웨이퍼 및 날짜별 Box Plot 생성 중...


### 📊 [10] VpiL 통계 분석 (VpiL Analysis)
* **파일명**: `VpiL_Analysis.py`
* **설명**: 산출된 $V_\pi\cdot L$ 데이터들의 웨이퍼 내 분포, 평균, 표준편차 등 전반적인 경향성과 수율을 분석합니다.

In [11]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter
from concurrent.futures import ThreadPoolExecutor

def q_sub(x, y):
    if len(x) < 3: return x[np.argmin(y)]
    idx = np.argmin(y)
    if idx == 0 or idx == len(x) - 1: return x[idx]
    c = np.polyfit(x[idx - 1:idx + 2], y[idx - 1:idx + 2], 2)
    return -c[1] / (2 * c[0]) if abs(c[0]) > 1e-12 else x[idx]


def _extract_vpil_data(args):
    d, L_length = args
    wafer, band, c, r = d['wafer_id'], d['band'], d['die_c'], d['die_r']
    radius = np.sqrt(c ** 2 + r ** 2)
    date_str = d.get('date', 'Unknown_Date')

    m = (d['ref_data']['wl'] >= d['wl_min']) & (d['ref_data']['wl'] <= d['wl_max'])
    v_ref_wl, v_ref_il = d['ref_data']['wl'][m], d['ref_data']['il'][m]

    if len(v_ref_wl) < 31:
        return None

    poly_func = np.poly1d(np.polyfit(v_ref_wl, savgol_filter(v_ref_il, 31, 3), 3))

    # 💡 [수정됨] 부동소수점 오차 방지를 위해 == 0.0 대신 절대값 비교 사용
    z_data = next((b for b in d['bias_data_list'] if b['bias'] is not None and abs(b['bias']) < 1e-3), None)
    if not z_data:
        return None

    m0 = (z_data['wl'] >= d['wl_min']) & (z_data['wl'] <= d['wl_max'])
    w0, i0 = z_data['wl'][m0], z_data['il'][m0]
    flat0 = savgol_filter(i0, 31, 3) - poly_func(w0)
    v0, _ = find_peaks(-flat0, prominence=0.3, distance=20)

    if len(v0) < 2:
        return None

    fsr = np.mean(np.diff(w0[v0]))
    cwl = w0[v0[np.argmin(np.abs(w0[v0] - d['target_wl']))]]

    volts, p_pi = [], []
    sh = fsr / 2.5
    for b in d['bias_data_list']:
        if b['bias'] is None: continue
        w, i = b['wl'], b['il']
        mb = (w >= d['wl_min']) & (w <= d['wl_max'])
        wb, ib = w[mb], i[mb]
        mloc = (wb >= cwl - sh) & (wb <= cwl + sh)
        if np.sum(mloc) < 5: continue
        flat = savgol_filter(ib[mloc], 11, 3) - poly_func(wb[mloc])
        vwl = q_sub(wb[mloc], flat)
        volts.append(b['bias'])
        p_pi.append(2.0 * (vwl - cwl) / fsr)

    if len(volts) < 5:
        return None

    volts, p_pi = np.array(volts), np.array(p_pi)

    fit = np.poly1d(np.polyfit(volts, p_pi, 2))
    slope_at_0 = max(np.abs(np.polyder(fit)(0.0)), 1e-5)
    vpil_0V = L_length / slope_at_0

    # 💡 데이터가 안 나온다면 여기가 원인일 수 있습니다! (범위 확인 필요)
    if not (0.1 <= vpil_0V <= 10.0):
        return None

    return {
        'Wafer': wafer, 'Band': band, 'Date': date_str, 'Column': c, 'Row': r,
        'Radius': radius, 'VpiL_0V': vpil_0V
    }


def main():
    zip_path = "./dat/HY202103"
    base_res_dir = "./res/png"
    target_wafers = ['D07', 'D08', 'D23', 'D24']
    L_length = 0.05
    TARGET_VPIL = {'LMZO': 1.4, 'LMZC': 2.0}

    wafer_map_dir = os.path.join(base_res_dir, "WaferMap")
    box_plot_dir = os.path.join(base_res_dir, "BoxPlot")
    os.makedirs(wafer_map_dir, exist_ok=True)
    os.makedirs(box_plot_dir, exist_ok=True)

    print("🚀 분석용 데이터 추출을 시작합니다...")

    parsed_data_list = list(parse_wafer_data(zip_path, target_wafers))
    tasks = [(d, L_length) for d in parsed_data_list]
    summary_rows = []

    with ThreadPoolExecutor(max_workers=8) as ex:
        futures = [ex.submit(_extract_vpil_data, t) for t in tasks]
        for f in futures:
            res = f.result()
            if res is not None:
                summary_rows.append(res)

    if not summary_rows:
        print("❌ 유효한 데이터를 찾지 못했습니다. _extract_vpil_data의 필터 조건을 확인하세요.")
        return

    df = pd.DataFrame(summary_rows)

    def merge_midnight_dates(date_val):
        date_str = str(date_val)
        if '0603' in date_str or '0604' in date_str:
            return '20190603'
        return date_str

    df['Date'] = df['Date'].apply(merge_midnight_dates)
    print("🕒 날짜 병합 처리 완료.")

    filtered_df = pd.DataFrame()
    df_grouped = df.groupby(['Wafer', 'Band', 'Date', 'Column', 'Row', 'Radius'], as_index=False)['VpiL_0V'].mean()

    for (wafer, band, date), group in df_grouped.groupby(['Wafer', 'Band', 'Date']):
        if len(group) > 5:
            m, s = group['VpiL_0V'].mean(), group['VpiL_0V'].std()
            valid_group = group[(group['VpiL_0V'] >= m - 3 * s) & (group['VpiL_0V'] <= m + 3 * s)]
            filtered_df = pd.concat([filtered_df, valid_group])
        else:
            filtered_df = pd.concat([filtered_df, group])

    max_r = filtered_df['Radius'].max()
    edge_limit = max_r * 0.75
    filtered_df['Region'] = np.where(filtered_df['Radius'] > edge_limit, 'Edge', 'Center')

    print(f"✅ 데이터 추출 및 필터링 완료! (총 {len(filtered_df)}개 다이 분석)")

    # Wafer Map 그리기

    print("▶ 웨이퍼 및 날짜별 Wafer Map 생성 중...")
    band_limits = {}
    for b in filtered_df['Band'].unique():
        b_data = filtered_df[filtered_df['Band'] == b]['VpiL_0V']
        band_limits[b] = {'min': b_data.min() - 0.05, 'max': b_data.max() + 0.05}

    # 격자 범위를 결정하기 위해 물리적 반경(`max_r`)을 기준으로 축 Limits 설정 (대칭)
    # 0.5 여유를 두어 원 가장자리 데이터가 격자에 걸치지 않게 함
    map_limit = np.ceil(max_r) + 0.5

    for (wafer, band, date), group in filtered_df.groupby(['Wafer', 'Band', 'Date']):
        # figsize를 정방형으로 유지
        plt.figure(figsize=(10, 10))
        ax = plt.gca()  # 현재 Axes 객체 가져오기

        # 1. 웨이퍼 외곽선 및 Edge 영역 구분선 (zorder=1: 맨 아래)
        theta = np.linspace(0, 2 * np.pi, 100)
        ax.plot((max_r + 0.5) * np.cos(theta), (max_r + 0.5) * np.sin(theta), color='#555555', lw=2, zorder=1)  # 외곽선
        ax.plot(edge_limit * np.cos(theta), edge_limit * np.sin(theta), color='#FF8888', ls='--', lw=2, alpha=0.7,
                zorder=1)  # Edge 구분

        # 2. 바둑판 격자 및 축 설정
        ax.set_aspect('equal')  # 정방형 비율 유지

        # 축 범위 설정 (0,0 기준 대칭)
        ax.set_xlim(-map_limit, map_limit)
        ax.set_ylim(-map_limit, map_limit)

        # Ticks 설정: 정수 좌표에 눈금을 매김 (zorder=0으로 설정하기 위해 grid 사용 권장)
        ticks = np.arange(-np.floor(map_limit), np.ceil(map_limit) + 1, 1)
        ax.set_xticks(ticks)
        ax.set_yticks(ticks)

        # 격자 추가: 주요 눈금(Major Ticks) 위에 회색 점선 격자 (zorder=2: 외곽선 위, 데이터 아래)
        ax.grid(True, which='major', axis='both', color='#DDDDDD', linestyle='--', linewidth=1, zorder=2)

        # 축 라벨 스타일 및 표시 설정 (plt.axis('off')를 제거했으므로 보임)
        ax.tick_params(axis='both', which='major', labelsize=11, labelcolor='#333333')
        # Ticks 굵게 (Optional)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontweight('bold')

        # 축 이름 설정
        ax.set_xlabel("Column (X Coordinate)", fontsize=14, fontweight='bold', labelpad=10)
        ax.set_ylabel("Row (Y Coordinate)", fontsize=14, fontweight='bold', labelpad=10)

        # 컬러 바 범위
        v_min, v_max = band_limits[band]['min'], band_limits[band]['max']

        # 3. 데이터 포인트 플로팅 (zorder=5: 격자 위)
        # 데이터가 많아 격자가 좁아 보일 경우 s(size)를 약간 줄여도 좋습니다 (예: 400).
        scatter = ax.scatter(group['Column'], group['Row'], c=group['VpiL_0V'], cmap='coolwarm',
                             vmin=v_min, vmax=v_max, s=500, edgecolor='black', alpha=0.9, zorder=5)

        # 4. 데이터 값 텍스트 표시 (zorder=6: 데이터 포인트 위)
        for _, row in group.iterrows():
            ax.text(row['Column'], row['Row'], f"{row['VpiL_0V']:.2f}",
                    ha='center', va='center', fontsize=9, weight='bold', color='black', zorder=6)

        # 컬러바 설정
        cb = plt.colorbar(scatter, shrink=0.8, pad=0.03)
        cb.set_label('Vpi*L @ 0V [V*cm]', fontsize=13, fontweight='bold')
        cb.ax.tick_params(labelsize=11)
        for label in cb.ax.yaxis.get_ticklabels(): label.set_weight("bold")

        # 제목 설정
        plt.title(f"Wafer Map: {wafer} / {band} (VpiL)\nDate: {date}", fontsize=17, fontweight='bold', pad=20)

        # [수정] plt.axis('off') 제거 (축을 보이게 함)
        # plt.tight_layout() # 격자와 라벨이 잘리지 않게 조정

        # 저장
        w_dir = os.path.join(wafer_map_dir, wafer, date)
        os.makedirs(w_dir, exist_ok=True)
        # bbox_inches='tight'는 축 라벨이 그림 밖으로 잘 나가지 않게 해줍니다.
        plt.savefig(os.path.join(w_dir, f"Map_{wafer}_{band}_{date}_VpiL_0V.png"), bbox_inches='tight', dpi=100)
        plt.close()
    # Box Plot 그리기 (VpiL 맞춤형 디자인)
    print("▶ 웨이퍼 및 날짜별 Box Plot 생성 중...")

    for (wafer, band, date), wbd_df in filtered_df.groupby(['Wafer', 'Band', 'Date']):
        current_target = TARGET_VPIL.get(band, 1.5)
        avg_vpil = wbd_df['VpiL_0V'].mean()

        plt.figure(figsize=(8, 8))
        pos, data, labels, colors = [], [], [], []

        c_data = wbd_df[wbd_df['Region'] == 'Center']['VpiL_0V'].values
        e_data = wbd_df[wbd_df['Region'] == 'Edge']['VpiL_0V'].values

        if len(c_data) > 0:
            pos.append(1)
            data.append(c_data)
            labels.append(f"Center\nn={len(c_data)}")
            colors.append('#3498db')
        if len(e_data) > 0:
            pos.append(2)
            data.append(e_data)
            labels.append(f"Edge\nn={len(e_data)}")
            colors.append('#e74c3c')

        if not data: continue

        all_data = np.concatenate(data)
        y_min = min(all_data.min(), current_target) - 0.2
        y_max = max(all_data.max(), current_target) + 0.2
        plt.ylim(y_min, y_max)

        # [VpiL 맞춤] 작을수록 좋으므로 Target 아래가 초록색(Good)
        plt.axhspan(y_min, current_target, facecolor='#e8f8f5', alpha=0.6, zorder=0, label='Good Region')
        plt.axhspan(current_target, y_max, facecolor='#fadbd8', alpha=0.6, zorder=0, label='Poor Region')

        box = plt.boxplot(data, positions=pos, patch_artist=True, widths=0.5,
                          flierprops=dict(marker='d', markerfacecolor='black', markersize=6, alpha=0.6),
                          zorder=2)
        for p, c in zip(box['boxes'], colors): p.set_facecolor(c); p.set_alpha(0.7)

        for p, d_arr in zip(pos, data):
            plt.scatter(np.random.normal(p, 0.05, len(d_arr)), d_arr, color='black', alpha=0.5, s=20, zorder=3)

        plt.axhline(avg_vpil, color='blue', ls='--', lw=2.5, label=f'Avg: {avg_vpil:.2f} V*cm', zorder=4)
        plt.axhline(current_target, color='red', ls='-', lw=2.5, label=f'Target: {current_target:.2f} V*cm', zorder=4)

        plt.title(f"VpiL Analysis : {wafer} ({band})\nDate: {date}", fontsize=18, fontweight='bold', pad=15)
        plt.xticks(pos, labels, fontsize=14, fontweight='bold')
        plt.yticks(fontsize=14, fontweight='bold')
        plt.ylabel("Vpi*L @ 0V [V*cm]", fontsize=16, fontweight='bold')

        plt.legend(loc='upper right', prop={'size': 11, 'weight': 'bold'})
        plt.grid(True, axis='y', ls=':', alpha=0.4, zorder=1)
        plt.xlim(0.5, max(pos) + 0.5)
        plt.tight_layout()

        box_dir = os.path.join(box_plot_dir, wafer, date)
        os.makedirs(box_dir, exist_ok=True)
        plt.savefig(os.path.join(box_dir, f"Box_{wafer}_{band}_{date}_VpiL_0V.png"), bbox_inches='tight')
        plt.close()

    print("✅ 모든 그래프가 웨이퍼별/날짜별 폴더 구조로 저장되었습니다!")
if __name__ == "__main__":
    main()

🚀 분석용 데이터 추출을 시작합니다...
🕒 날짜 병합 처리 완료.
✅ 데이터 추출 및 필터링 완료! (총 72개 다이 분석)
▶ 웨이퍼 및 날짜별 Wafer Map 생성 중...
▶ 웨이퍼 및 날짜별 Box Plot 생성 중...
✅ 모든 그래프가 웨이퍼별/날짜별 폴더 구조로 저장되었습니다!


### 🗺️ [11] 통합 시각화: Wafer Map & Box Plot (Combine Plot)
* **파일명**: `combine_plot.py`
* **설명**: 추출된 IL, ER, Vpi 데이터를 기반으로 **Wafer Map**을 그려 수율의 공간적 분포를 확인하고, **Box Plot**을 통해 전체적인 산포와 다이(Die) 간 편차를 한눈에 비교합니다.

In [12]:
import os
import math
from PIL import Image


def get_sort_index(filename):
    """
    다이(Die) 분석 이미지의 정렬 순서를 결정합니다.
    1.Plot(Raw) -> 2.Flatting -> 3.Fitting -> 4.Zoom -> 5.Phase shift -> 6.VpiL
    """
    name = filename.lower()
    if 'raw' in name or 'plot' in name:
        return 1
    elif 'flatting' in name or 'flat' in name:
        return 2
    elif 'zoom' in name:
        return 3
    elif 'fitting' in name or 'fit' in name:
        return 4
    elif 'phase' in name:
        return 5
    elif 'vpil' in name:
        return 6
    else:
        return 99


def get_map_sort_index(filename):
    """
    웨이퍼 맵 및 박스 플롯 이미지의 정렬 순서를 결정합니다.
    요청하신 순서: 1.ER(소광비) -> 2.IL(손실) -> 3.VpiL(효율) 순서로 배치합니다.
    """
    name = filename.upper()
    if 'ER' in name:
        return 1
    elif 'IL' in name and 'VPIL' not in name:
        return 2
    elif 'VPIL' in name:
        return 3
    else:
        return 99


def merge_die_images(base_dir, delete_originals=True):
    """
    각 날짜 폴더 내의 분석 이미지들을 좌표(Die) 및 밴드(Band)별로 그룹화하여
    가로 3, 세로 2 격자 형태로 병합하고 원본을 삭제합니다.
    """
    print("▶ 1. 날짜 폴더 내 다이(Die) 분석 이미지 병합 및 원본 파일 관리를 시작합니다...")
    combine_count = 0
    deleted_count = 0

    for wafer in os.listdir(base_dir):
        wafer_path = os.path.join(base_dir, wafer)

        # 🌟 WaferMap, BoxPlot 등 시스템 폴더는 다이(Die) 병합에서 제외
        if not os.path.isdir(wafer_path) or wafer in ["Analysis", "WaferMap", "BoxPlot"]:
            continue

        for date_folder in os.listdir(wafer_path):
            date_path = os.path.join(wafer_path, date_folder)
            if not os.path.isdir(date_path):
                continue

            for f in os.listdir(date_path):
                if f.startswith('HY202103_') and 'LION1_DCM' in f and f.endswith('.png'):
                    try:
                        os.remove(os.path.join(date_path, f))
                    except:
                        pass

            image_files = [f for f in os.listdir(date_path) if f.endswith('.png')
                           and not f.startswith('Map_')
                           and not f.startswith('Box_')
                           and not f.startswith('HY202103_')]

            if not image_files:
                continue

            die_groups = {}
            for f in image_files:
                parts = f.replace(".png", "").split('_')
                if len(parts) >= 4:
                    try:
                        c_val = parts[1].replace('C', '')
                        r_val = parts[2].replace('R', '')
                        band = parts[3]
                        group_key = (c_val, r_val, band)

                        if group_key not in die_groups:
                            die_groups[group_key] = []
                        die_groups[group_key].append(f)
                    except Exception as e:
                        print(f"파일 이름 분석 오류: {f} -> {e}")

            for (c_val, r_val, band), files in die_groups.items():
                if not files:
                    continue

                files.sort(key=get_sort_index)

                images = []
                valid_files = []
                for img_name in files:
                    try:
                        img_path = os.path.join(date_path, img_name)
                        img = Image.open(img_path)
                        img.load()
                        images.append(img)
                        valid_files.append(img_path)
                    except:
                        pass
                if not images:
                    continue

                cols = 3
                rows = math.ceil(len(images) / cols)
                max_width = max(img.size[0] for img in images)
                max_height = max(img.size[1] for img in images)

                grid_width = cols * max_width
                grid_height = rows * max_height
                new_im = Image.new('RGB', (grid_width, grid_height), color='white')

                for i, img in enumerate(images):
                    x_offset = (i % cols) * max_width
                    y_offset = (i // cols) * max_height
                    new_im.paste(img, (x_offset, y_offset))
                    img.close()

                save_filename = f"HY202103_{wafer}_({c_val},{r_val})_LION1_DCM_{band}.png"
                new_im.save(os.path.join(date_path, save_filename))
                combine_count += 1

                if delete_originals:
                    for orig_path in valid_files:
                        try:
                            os.remove(orig_path)
                            deleted_count += 1
                        except Exception as e:
                            print(f"  [경고] 원본 파일 삭제 실패 ({os.path.basename(orig_path)}): {e}")

    print(f"  ✅ 총 {combine_count}개의 다이(Die) 요약 이미지 병합 완료!")
    if delete_originals:
        print(f"  🧹 병합에 사용된 원본 이미지 총 {deleted_count}개 삭제 완료.\n")


def merge_wafer_maps(base_dir, delete_originals=True):
    """🌟 WaferMap 폴더 내의 웨이퍼 맵 3종(ER, IL, VpiL)을 1개의 이미지로 가로 병합합니다."""
    wafermap_dir = os.path.join(base_dir, "WaferMap")
    if not os.path.exists(wafermap_dir):
        print(f"  [안내] WaferMap 폴더가 없어 웨이퍼 맵 병합을 건너뜁니다.")
        return

    print("▶ 2. 웨이퍼별 통합 맵(ER, IL, VpiL) 병합 및 원본 파일 관리를 시작합니다...")
    map_combine_count = 0
    deleted_count = 0

    for wafer in os.listdir(wafermap_dir):
        w_path = os.path.join(wafermap_dir, wafer)
        if not os.path.isdir(w_path):
            continue

        for date_folder in os.listdir(w_path):
            d_path = os.path.join(w_path, date_folder)
            if not os.path.isdir(d_path):
                continue

            band_images = {}
            for f in os.listdir(d_path):
                if f.startswith('Summary_WaferMap'):
                    try:
                        os.remove(os.path.join(d_path, f))
                    except:
                        pass
                    continue

                if f.startswith('Map_') and f.endswith('.png'):
                    # 파일명 예: Map_D07_LMZO_20190715_IL.png
                    parts = f.replace(f"Map_{wafer}_", "").split("_")
                    band = parts[0]
                    if band not in band_images:
                        band_images[band] = []
                    band_images[band].append(f)

            for band, files in band_images.items():
                if len(files) < 2:
                    continue

                files.sort(key=get_map_sort_index)

                images = []
                valid_files = []
                for f in files:
                    try:
                        img_path = os.path.join(d_path, f)
                        img = Image.open(img_path)
                        img.load()
                        images.append(img)
                        valid_files.append(img_path)
                    except:
                        pass

                if not images:
                    continue

                cols = len(images)
                max_width = max(img.size[0] for img in images)
                max_height = max(img.size[1] for img in images)

                new_im = Image.new('RGB', (max_width * cols, max_height), color='white')
                for i, img in enumerate(images):
                    new_im.paste(img, (i * max_width, 0))
                    img.close()

                save_filename = f"Summary_WaferMap_{wafer}_{band}_{date_folder}.png"
                new_im.save(os.path.join(d_path, save_filename))
                map_combine_count += 1

                if delete_originals:
                    for orig_path in valid_files:
                        try:
                            os.remove(orig_path)
                            deleted_count += 1
                        except Exception as e:
                            print(f"  [경고] 웨이퍼 맵 원본 삭제 실패 ({os.path.basename(orig_path)}): {e}")

    print(f"  ✅ 총 {map_combine_count}장의 통합 웨이퍼 맵 생성 완료!")
    if delete_originals:
        print(f"  🧹 병합에 사용된 웨이퍼 맵 원본 이미지 총 {deleted_count}개 삭제 완료.\n")


def merge_box_plots(base_dir, delete_originals=True):
    """🌟 BoxPlot 폴더 내의 박스 플롯 3종(ER, IL, VpiL)을 1개의 이미지로 가로 병합합니다."""
    boxplot_dir = os.path.join(base_dir, "BoxPlot")
    if not os.path.exists(boxplot_dir):
        print(f"  [안내] BoxPlot 폴더가 없어 박스 플롯 병합을 건너뜁니다.")
        return

    print("▶ 3. 통합 박스 플롯(ER, IL, VpiL) 병합 및 원본 파일 관리를 시작합니다...")
    box_combine_count = 0
    deleted_count = 0

    for wafer in os.listdir(boxplot_dir):
        w_path = os.path.join(boxplot_dir, wafer)
        if not os.path.isdir(w_path):
            continue

        for date_folder in os.listdir(w_path):
            d_path = os.path.join(w_path, date_folder)
            if not os.path.isdir(d_path):
                continue

            band_images = {}
            for f in os.listdir(d_path):
                if f.startswith('Summary_BoxPlot'):
                    try:
                        os.remove(os.path.join(d_path, f))
                    except:
                        pass
                    continue

                if f.startswith('Box_') and f.endswith('.png'):
                    # 파일명 예: Box_D07_LMZO_20190715_IL.png
                    parts = f.replace(f"Box_{wafer}_", "").split("_")
                    band = parts[0]
                    if band not in band_images:
                        band_images[band] = []
                    band_images[band].append(f)

            for band, files in band_images.items():
                if len(files) < 2:
                    continue

                files.sort(key=get_map_sort_index)

                images = []
                valid_files = []
                for f in files:
                    try:
                        img_path = os.path.join(d_path, f)
                        img = Image.open(img_path)
                        img.load()
                        images.append(img)
                        valid_files.append(img_path)
                    except:
                        pass

                if not images:
                    continue

                cols = len(images)
                max_width = max(img.size[0] for img in images)
                max_height = max(img.size[1] for img in images)

                new_im = Image.new('RGB', (max_width * cols, max_height), color='white')
                for i, img in enumerate(images):
                    new_im.paste(img, (i * max_width, 0))
                    img.close()

                save_filename = f"Summary_BoxPlot_{wafer}_{band}_{date_folder}.png"
                new_im.save(os.path.join(d_path, save_filename))
                box_combine_count += 1

                if delete_originals:
                    for orig_path in valid_files:
                        try:
                            os.remove(orig_path)
                            deleted_count += 1
                        except Exception as e:
                            print(f"  [경고] 박스 플롯 원본 삭제 실패 ({os.path.basename(orig_path)}): {e}")

    print(f"  ✅ 총 {box_combine_count}장의 통합 박스 플롯 생성 완료!")
    if delete_originals:
        print(f"  🧹 병합에 사용된 박스 플롯 원본 이미지 총 {deleted_count}개 삭제 완료.\n")


def main():
    base_dir = "./res/png"
    base_dir = os.path.abspath(base_dir)

    if not os.path.exists(base_dir):
        print(f"❌ [오류] {base_dir} 경로를 찾을 수 없습니다.")
        return

    # 🌟 원본 파일을 실제로 삭제하려면 True, 원본을 유지하고 싶다면 False로 설정하세요.
    DELETE_ORIGINAL_FILES = True

    # 1. Die 이미지 병합 실행
    merge_die_images(base_dir, delete_originals=DELETE_ORIGINAL_FILES)

    # 2. Wafer Map 이미지 병합 실행 (새 경로 적용)
    merge_wafer_maps(base_dir, delete_originals=DELETE_ORIGINAL_FILES)

    # 3. Box Plot 이미지 병합 실행 (새로 추가)
    merge_box_plots(base_dir, delete_originals=DELETE_ORIGINAL_FILES)

    print("\n🎉 모든 이미지 병합 및 원본 파일 정리 작업이 성공적으로 마무리되었습니다!")
main()

▶ 1. 날짜 폴더 내 다이(Die) 분석 이미지 병합 및 원본 파일 관리를 시작합니다...
  ✅ 총 98개의 다이(Die) 요약 이미지 병합 완료!
  🧹 병합에 사용된 원본 이미지 총 563개 삭제 완료.

▶ 2. 웨이퍼별 통합 맵(ER, IL, VpiL) 병합 및 원본 파일 관리를 시작합니다...
  ✅ 총 7장의 통합 웨이퍼 맵 생성 완료!
  🧹 병합에 사용된 웨이퍼 맵 원본 이미지 총 21개 삭제 완료.

▶ 3. 통합 박스 플롯(ER, IL, VpiL) 병합 및 원본 파일 관리를 시작합니다...
  ✅ 총 7장의 통합 박스 플롯 생성 완료!
  🧹 병합에 사용된 박스 플롯 원본 이미지 총 21개 삭제 완료.


🎉 모든 이미지 병합 및 원본 파일 정리 작업이 성공적으로 마무리되었습니다!


### 💾 [12] 분석 결과 요약 및 내보내기 (Export Summary)
* **파일명**: `export_summary.py`
* **설명**: 웨이퍼 내 모든 소자의 분석 결과(IL, ER, Vpi 등)를 종합하여 CSV 또는 Excel 형태의 최종 레포트 파일로 저장합니다.

In [13]:
import os
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, savgol_filter
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

# [설정] 아웃라이어(Outlier) 및 에러 판단 기준

THRESHOLD = {
    'IL_MIN': -20.0,
    'ER_MIN': 10.0,
    'VPIL_MIN': 0.2,
    'VPIL_MAX': 3.0
}

zip_path = "./dat/HY202103"

# 저장 디렉토리 분리
save_dir_xlsx = "./res/xlsx"
save_dir_csv = "./res/csv"
os.makedirs(save_dir_xlsx, exist_ok=True)
os.makedirs(save_dir_csv, exist_ok=True)

target_wafers = ['D07', 'D08', 'D23', 'D24']
L_length = 0.05


def q_sub(x, y):
    if len(x) < 3: return x[np.argmin(y)]
    idx = np.argmin(y)
    if idx == 0 or idx == len(x) - 1: return x[idx]
    c = np.polyfit(x[idx - 1:idx + 2], y[idx - 1:idx + 2], 2)
    return -c[1] / (2 * c[0]) if abs(c[0]) > 1e-12 else x[idx]


def check_status(row):
    reasons = []
    if pd.isna(row['IL_dB']):
        reasons.append("IL 데이터 없음")
    elif row['IL_dB'] < THRESHOLD['IL_MIN']:
        reasons.append(f"IL 손실 과다(<{THRESHOLD['IL_MIN']})")

    if pd.isna(row['ER_dB']):
        reasons.append("ER 데이터 없음")
    elif row['ER_dB'] < THRESHOLD['ER_MIN']:
        reasons.append(f"ER 낮음(<{THRESHOLD['ER_MIN']})")

    if pd.isna(row['VpiL_Vcm']):
        reasons.append("VpiL 계산 실패")
    elif row['VpiL_Vcm'] < THRESHOLD['VPIL_MIN'] or row['VpiL_Vcm'] > THRESHOLD['VPIL_MAX']:
        reasons.append(f"VpiL 범위 초과({THRESHOLD['VPIL_MIN']}~{THRESHOLD['VPIL_MAX']})")

    return ("정상", "") if not reasons else ("이상 발생", ", ".join(reasons))


# 엑셀 서식 자동 적용 함수

def apply_excel_style(worksheet, dataframe):
    header_fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
    header_font = Font(color='FFFFFF', bold=True)
    error_fill = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    link_font = Font(color='0563C1', underline='single')
    border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'),
                    bottom=Side(style='thin'))

    col_idx = {col: i for i, col in enumerate(dataframe.columns, 1)}

    for col_num, column_title in enumerate(dataframe.columns, 1):
        cell = worksheet.cell(row=1, column=col_num)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center')

        for row_num in range(2, len(dataframe) + 2):
            data_cell = worksheet.cell(row=row_num, column=col_num)
            data_cell.border = border

            # 1. Folder_Link 열 (개별 다이 통합 PNG 열기)
            if column_title == 'Folder_Link':
                target_image = data_cell.value
                if target_image:
                    data_cell.value = f'=HYPERLINK("{target_image}", "🖼️ 개별 다이 이미지 보기")'
                    data_cell.font = link_font
                    data_cell.alignment = Alignment(horizontal='center')

            # 2. 웨이퍼 맵 하이퍼링크 (웨이퍼 맵 통합 이미지 열기)
            elif column_title == 'Map_Image_Link':
                target_image = data_cell.value
                if target_image:
                    data_cell.value = f'=HYPERLINK("{target_image}", "🗺️ 웨이퍼 맵 통합 이미지 보기")'
                    data_cell.font = link_font
                    data_cell.alignment = Alignment(horizontal='center')

            # 3. 박스 플롯 하이퍼링크 (박스 플롯 통합 이미지 열기)
            elif column_title == 'BoxPlot_Image_Link':
                target_image = data_cell.value
                if target_image:
                    data_cell.value = f'=HYPERLINK("{target_image}", "📊 박스 플롯 통합 이미지 보기")'
                    data_cell.font = link_font
                    data_cell.alignment = Alignment(horizontal='center')

            else:
                if column_title in ['IL_dB', 'ER_dB', 'VpiL_Vcm'] and isinstance(data_cell.value, (int, float)):
                    data_cell.number_format = '0.000'
                elif column_title == 'R_Squared' and isinstance(data_cell.value, (int, float)):
                    data_cell.number_format = '0.0000'

            # 에러 행 하이라이트
            if 'Status' in col_idx:
                status_value = worksheet.cell(row=row_num, column=col_idx['Status']).value
                if status_value == "이상 발생" and column_title not in ['Folder_Link', 'Map_Image_Link', 'BoxPlot_Image_Link']:
                    data_cell.fill = error_fill

    for col in worksheet.columns:
        max_length = 0
        column_letter = get_column_letter(col[0].column)
        for cell in col:
            if cell.value is not None:
                text = str(cell.value)
                max_length = max(max_length, sum(2 if ord(c) > 127 else 1 for c in text))
        worksheet.column_dimensions[column_letter].width = min(max_length + 3, 50)
# 메인 실행 블록

print("🚀 데이터 분석 및 계산을 시작합니다...")
data_list = []

for d in parse_wafer_data(zip_path, target_wafers):
    wafer, band, c, r = d['wafer_id'], d['band'], d['die_c'], d['die_r']
    radius = np.sqrt(c ** 2 + r ** 2)
    date_str = d.get('date', 'Unknown_Date')

    m = (d['ref_data']['wl'] >= d['wl_min']) & (d['ref_data']['wl'] <= d['wl_max'])
    v_ref_wl, v_ref_il = d['ref_data']['wl'][m], d['ref_data']['il'][m]
    if len(v_ref_wl) < 31: continue

    sm_ref = savgol_filter(v_ref_il, 31, 3)
    poly_func = np.poly1d(np.polyfit(v_ref_wl, sm_ref, 3))
    y_fitted = poly_func(v_ref_wl)

    ss_res = np.sum((sm_ref - y_fitted) ** 2)
    ss_tot = np.sum((sm_ref - np.mean(sm_ref)) ** 2)
    r_squared = 1.0 if ss_tot == 0 else 1 - (ss_res / ss_tot)

    max_peak = float('-inf')
    for b in d['bias_data_list']:
        if b['bias'] is None: continue
        mb = (b['wl'] >= d['wl_min']) & (b['wl'] <= d['wl_max'])
        if np.any(mb):
            max_peak = max(max_peak, np.max(b['il'][mb]))
    il_val = max_peak if max_peak != float('-inf') else np.nan

    max_er = 0.0
    for b in d['bias_data_list']:
        if b['bias'] is None: continue
        mb = (b['wl'] >= d['wl_min']) & (b['wl'] <= d['wl_max'])
        v_wl, v_il = b['wl'][mb], b['il'][mb]
        if len(v_wl) == 0: continue
        flat_il = v_il - poly_func(v_wl)
        peaks, _ = find_peaks(flat_il, prominence=3.0, distance=30)
        final_il = flat_il - np.poly1d(np.polyfit(v_wl[peaks], flat_il[peaks], 1))(v_wl) if len(peaks) >= 2 else flat_il
        max_er = max(max_er, np.percentile(final_il, 99) - np.percentile(final_il, 1))
    er_val = max_er if max_er > 0 else np.nan

    vpil_val = np.nan
    z_data = next((b for b in d['bias_data_list'] if b['bias'] == 0.0), None)
    if z_data:
        m0 = (z_data['wl'] >= d['wl_min']) & (z_data['wl'] <= d['wl_max'])
        w0, i0 = z_data['wl'][m0], z_data['il'][m0]
        v0, _ = find_peaks(-(savgol_filter(i0, 31, 3) - poly_func(w0)), prominence=0.3, distance=20)
        if len(v0) >= 2:
            fsr, cwl = np.mean(np.diff(w0[v0])), w0[v0[np.argmin(np.abs(w0[v0] - d['target_wl']))]]
            volts, p_pi = [], []
            for b in d['bias_data_list']:
                if b['bias'] is None: continue
                w, i = b['wl'], b['il']
                mb = (w >= d['wl_min']) & (w <= d['wl_max'])
                wb, ib = w[mb], i[mb]
                mloc = (wb >= cwl - fsr / 2.5) & (wb <= cwl + fsr / 2.5)
                if np.sum(mloc) >= 5:
                    volts.append(b['bias'])
                    p_pi.append(
                        2.0 * (q_sub(wb[mloc], savgol_filter(ib[mloc], 11, 3) - poly_func(wb[mloc])) - cwl) / fsr)
            if len(volts) >= 5:
                vpil_0V = L_length / max(np.abs(np.polyder(np.poly1d(np.polyfit(volts, p_pi, 2)))(0.0)), 1e-5)
                if 0.1 <= vpil_0V <= 10.0: vpil_val = vpil_0V

    data_list.append({
        'Wafer': wafer, 'Band': band, 'Date': date_str, 'Column': c, 'Row': r, 'Radius': radius,
        'IL_dB': il_val, 'ER_dB': er_val, 'VpiL_Vcm': vpil_val, 'R_Squared': r_squared
    })

# --- 전체 데이터 정리 및 날짜 병합 ---
df_full = pd.DataFrame(data_list).groupby(['Wafer', 'Band', 'Date', 'Column', 'Row', 'Radius'], as_index=False).min(
    numeric_only=True)
df_full['Status'], df_full['Reason'] = zip(*df_full.apply(check_status, axis=1))


# [추가된 부분] 0603, 0604 데이터 병합
def merge_midnight_dates(date_val):
    date_str = str(date_val)
    if '0603' in date_str or '0604' in date_str:
        return '20190603'
    return date_str


df_full['Date'] = df_full['Date'].apply(merge_midnight_dates)
print("🕒 0603 및 0604 날짜 데이터를 '20190603'로 통합했습니다.")


# 🌟 1. 개별 다이 이미지 경로 생성
def generate_die_paths(row):
    wafer = str(row['Wafer'])
    date = str(row['Date'])
    c = int(row['Column'])
    r = int(row['Row'])
    band = str(row['Band'])
    base_dir = f"..\\png\\{wafer}\\{date}"
    image_name = f"HY202103_{wafer}_({c},{r})_LION1_DCM_{band}.png"
    return f"{base_dir}\\{image_name}"


df_full['Folder_Link'] = df_full.apply(generate_die_paths, axis=1)

print("--------------------------------------------------")
print("▶ 결과 파일 저장을 시작합니다...")

# [순수 CSV 파일 생성] (링크 컬럼 제거됨, 합본 포함)

csv_df = df_full.drop(columns=['Folder_Link'], errors='ignore')

for wafer_id in csv_df['Wafer'].unique():
    wafer_csv = csv_df[csv_df['Wafer'] == wafer_id]
    csv_file_path = os.path.join(save_dir_csv, f"{wafer_id}_Process_result.csv")
    wafer_csv.to_csv(csv_file_path, index=False, encoding='utf-8-sig')
    print(f"  - 개별 순수 CSV 저장 완료: {csv_file_path}")

total_csv_path = os.path.join(save_dir_csv, "Total_Process_result.csv")
csv_df.to_csv(total_csv_path, index=False, encoding='utf-8-sig')
print(f"  - 🌟 전체 통합 CSV 저장 완료: {total_csv_path}")

print("--------------------------------------------------")

# [엑셀 XLSX 파일 생성] (병합 이미지 링크 포함)

for wafer_id in df_full['Wafer'].unique():
    wafer_df = df_full[df_full['Wafer'] == wafer_id].copy()
    wafer_file_path = os.path.join(save_dir_xlsx, f"{wafer_id}_Process_result.xlsx")
    with pd.ExcelWriter(wafer_file_path, engine='openpyxl') as writer:
        wafer_df.to_excel(writer, index=False, sheet_name='Analysis_Result')
        apply_excel_style(writer.sheets['Analysis_Result'], wafer_df)
    print(f"  - 개별 XLSX 저장 완료: {wafer_file_path}")

total_xlsx_path = os.path.join(save_dir_xlsx, "Total_Process_result.xlsx")
with pd.ExcelWriter(total_xlsx_path, engine='openpyxl') as writer:
    df_full.to_excel(writer, index=False, sheet_name='Analysis_Result')
    apply_excel_style(writer.sheets['Analysis_Result'], df_full)
print(f"  - 🌟 전체 통합 XLSX 저장 완료: {total_xlsx_path}")

# 3. 새로운 Analysis.xlsx 및 Analysis.csv 생성 (통합 이미지 직접 링크)

df_index = df_full[['Wafer', 'Date', 'Band']].drop_duplicates().reset_index(drop=True)

# 3. 새로운 Analysis.xlsm 및 Analysis.csv 생성 (통합 이미지 직접 링크)
df_index = df_full[['Wafer', 'Date', 'Band']].drop_duplicates().reset_index(drop=True)

# 기준이 되는 현재 프로젝트의 절대 경로를 미리 구해옵니다.
BASE_ABS_PATH = os.path.abspath("..")  # 스크립트 상위인 PE2 폴더 기준


# 🌟 1. 개별 다이 이미지 절대 경로 생성
def generate_die_paths(row):
    wafer = str(row['Wafer'])
    date = str(row['Date'])
    c = int(row['Column'])
    r = int(row['Row'])
    band = str(row['Band'])

    # 조립된 상대 경로: ../res/png/{wafer}/{date}/...
    rel_path = f"res/png/{wafer}/{date}/HY202103_{wafer}_({c},{r})_LION1_DCM_{band}.png"
    abs_path = os.path.join(BASE_ABS_PATH, rel_path).replace('\\', '/')

    return f"file:///{abs_path}"


# 🌟 2. 웨이퍼 맵 통합 파일 절대 경로 생성
def generate_map_paths(row):
    wafer = str(row['Wafer'])
    date = str(row['Date'])
    band = str(row['Band'])

    rel_path = f"res/png/WaferMap/{wafer}/{date}/Summary_WaferMap_{wafer}_{band}_{date}.png"
    abs_path = os.path.join(BASE_ABS_PATH, rel_path).replace('\\', '/')

    return f"file:///{abs_path}"


# 🌟 3. 박스 플롯 통합 파일 절대 경로 생성
def generate_boxplot_paths(row):
    wafer = str(row['Wafer'])
    date = str(row['Date'])
    band = str(row['Band'])

    rel_path = f"res/png/BoxPlot/{wafer}/{date}/Summary_BoxPlot_{wafer}_{band}_{date}.png"
    abs_path = os.path.join(BASE_ABS_PATH, rel_path).replace('\\', '/')

    return f"file:///{abs_path}"


# --- 경로 함수를 df_full에 적용 ---
df_full['Folder_Link'] = df_full.apply(generate_die_paths, axis=1)

print("--------------------------------------------------")
print("▶ 결과 파일 저장을 시작합니다...")

# [순수 CSV 파일 생성] (링크 컬럼 제거됨, 합본 포함)
csv_df = df_full.drop(columns=['Folder_Link'], errors='ignore')

for wafer_id in csv_df['Wafer'].unique():
    wafer_csv = csv_df[csv_df['Wafer'] == wafer_id]
    csv_file_path = os.path.join(save_dir_csv, f"{wafer_id}_Process_result.csv")
    wafer_csv.to_csv(csv_file_path, index=False, encoding='utf-8-sig')
    print(f"  - 개별 순수 CSV 저장 완료: {csv_file_path}")

total_csv_path = os.path.join(save_dir_csv, "Total_Process_result.csv")
csv_df.to_csv(total_csv_path, index=False, encoding='utf-8-sig')
print(f"  - 🌟 전체 통합 CSV 저장 완료: {total_csv_path}")

print("--------------------------------------------------")

# [엑셀 XLSX 파일 생성] (병합 이미지 링크 포함)
for wafer_id in df_full['Wafer'].unique():
    wafer_df = df_full[df_full['Wafer'] == wafer_id].copy()
    wafer_file_path = os.path.join(save_dir_xlsx, f"{wafer_id}_Process_result.xlsx")
    with pd.ExcelWriter(wafer_file_path, engine='openpyxl') as writer:
        wafer_df.to_excel(writer, index=False, sheet_name='Analysis_Result')
        apply_excel_style(writer.sheets['Analysis_Result'], wafer_df)
    print(f"  - 개별 XLSX 저장 완료: {wafer_file_path}")

total_xlsx_path = os.path.join(save_dir_xlsx, "Total_Process_result.xlsx")
with pd.ExcelWriter(total_xlsx_path, engine='openpyxl') as writer:
    df_full.to_excel(writer, index=False, sheet_name='Analysis_Result')
    apply_excel_style(writer.sheets['Analysis_Result'], df_full)
print(f"  - 🌟 전체 통합 XLSX 저장 완료: {total_xlsx_path}")

# 3. 새로운 Analysis.xlsm 및 Analysis.csv 생성 (통합 이미지 직접 링크)
df_index = df_full[['Wafer', 'Date', 'Band']].drop_duplicates().reset_index(drop=True)

# 링크 적용 (절대경로 및 file:/// 규격 반영 함수 대입)
df_index['Map_Image_Link'] = df_index.apply(generate_map_paths, axis=1)
df_index['BoxPlot_Image_Link'] = df_index.apply(generate_boxplot_paths, axis=1)

# Analysis.csv 저장 (경로 제거)
analysis_csv_path = os.path.join(save_dir_csv, "Analysis.csv")
df_index.drop(columns=['Map_Image_Link', 'BoxPlot_Image_Link'], errors='ignore').to_csv(analysis_csv_path,
                                                                                        index=False,
                                                                                        encoding='utf-8-sig')
print(f"  - 🌟 마스터 CSV 저장 완료: {analysis_csv_path}")

# Analysis.xlsx 저장 (대시보드 형태)
total_file_path = os.path.join(save_dir_xlsx, "Analysis.xlsx")
with pd.ExcelWriter(total_file_path, engine='openpyxl') as writer:
    df_index.to_excel(writer, index=False, sheet_name='Dashboard_Links')
    apply_excel_style(writer.sheets['Dashboard_Links'], df_index)
print(f"  - 🌟 마스터 XLSX 저장 완료 (통합 이미지 다이렉트 링크 연결 완료): {total_file_path}")

🚀 데이터 분석 및 계산을 시작합니다...
🕒 0603 및 0604 날짜 데이터를 '20190603'로 통합했습니다.
--------------------------------------------------
▶ 결과 파일 저장을 시작합니다...
  - 개별 순수 CSV 저장 완료: ./res/csv\D07_Process_result.csv
  - 개별 순수 CSV 저장 완료: ./res/csv\D08_Process_result.csv
  - 개별 순수 CSV 저장 완료: ./res/csv\D23_Process_result.csv
  - 개별 순수 CSV 저장 완료: ./res/csv\D24_Process_result.csv
  - 🌟 전체 통합 CSV 저장 완료: ./res/csv\Total_Process_result.csv
--------------------------------------------------
  - 개별 XLSX 저장 완료: ./res/xlsx\D07_Process_result.xlsx
  - 개별 XLSX 저장 완료: ./res/xlsx\D08_Process_result.xlsx
  - 개별 XLSX 저장 완료: ./res/xlsx\D23_Process_result.xlsx
  - 개별 XLSX 저장 완료: ./res/xlsx\D24_Process_result.xlsx


PermissionError: [Errno 13] Permission denied: './res/xlsx\\Total_Process_result.xlsx'